# Time-dependent cholera transmission in Yemen's second wave: DPINN against a Fourier-feature PINN

This notebook is self-contained. The surveillance record it fits is embedded below as a compressed
blob, so it runs with no network access and no external files. Nothing in it uses a synthetic twin
or a known parameter vector: every error reported is a fitted quantity measured against the
**observed** incidence, and every parameter panel compares architectures against one another.

**What it does.**

1. Unpacks the official Yemen cholera record (WHO/UNICEF daily bulletins, 2017-05-27 to 2018-02-18, day 0 = 2017-05-22)
   and the governorate coordinate file built from the IASC Common Operational Dataset boundaries.
2. Draws the observed incidence field $I(t,x)$ over the 21 reporting governorates.
3. At each of four major governorates of the second wave, infers the time-dependent transmission
   rates $\beta_1(t)$ (human route) and $\beta_2(t)$ (environmental route) with three methods:
   a deterministic Poisson-deviance fit under a hard PDE constraint, a **dual-network PINN**, and a
   **Fourier-feature PINN**.
4. Reports MAE, MSE and RMSE of each fitted surface against the observed record, the temporal
   reproduction number $\mathcal{R}_0(t)$ from each, and the comparison between them.

**The model.** An age-and-path structured SEIP transport-diffusion system on $(t,x)$, with
$E(t,a,x)$ resolved by time since infection ($a \le A_{\max}=7$ d) and $P(t,b,x)$ by time since
shedding ($b \le B_{\max}=30$ d). Transmission enters at exactly one place, the non-local renewal
condition at $a=0$:

$$E(t,0,x) \;=\; S(t,x)\left[\int_0^{A_{\max}}\!\beta_1(t)\,f(a)\,I(t,a,x)\,da \;+\; \int_0^{B_{\max}}\!\beta_2(t)\,h(b)\,P(t,b,x)\,db\right].$$

That single point of entry is why the coupling weight on the renewal residual is reported rather
than left at a default: it is the only relation tying the parameter network to the state network.

## 0. Environment

In [ ]:
import sys, subprocess
for pkg in ("torch", "scipy", "pandas", "matplotlib", "nbformat"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "--break-system-packages", pkg], check=True)
import numpy as np, pandas as pd, torch, matplotlib
print("numpy", np.__version__, "| torch", torch.__version__, "| pandas", pd.__version__)
torch.set_num_threads(2)

## 1. The dataset, embedded

Three files are unpacked here.

| file | what it is |
|---|---|
| `cases_long.csv` | the harmonized panel: one row per governorate per bulletin interval, carrying the cumulative count, the new cases in the interval, the interval length in days, the population denominator and the per-capita incidence rate |
| `raw_population.csv` | the governorate population denominators |
| `governorate_coords.csv` | the spatial coordinate $x_g$, great-circle distance from Sana'a |

**Provenance.** The counts come from the WHO/UNICEF Yemen cholera daily epidemiological bulletins
as republished on the Humanitarian Data Exchange (`data.humdata.org`, *Yemen Cholera Outbreak
Daily Epidemiology Update*). The coordinate file is built from the IASC Common Operational Dataset
administrative boundaries for Yemen, `COD-AB-YEM` v01 (`valid_on` 2019-11-22): the area-weighted
planar centroid of each governorate polygon in WGS84, with $x=0$ at the Amanat Al Asimah (Sana'a
City) centroid and distance by haversine at $R=6371.0088$ km.

Two properties of the record that matter downstream, and are checked rather than asserted below:
the per-governorate reporting gap `interval_days` is not constant, so the fit compares interval
*averages* rather than instantaneous rates; and the two Hadramaut district sub-reports (`Moklla`,
`Say'on`) carry no governorate p-code in the source file, which is why the record holds 20 distinct
p-codes but 21 reporting governorates.

In [ ]:
DATA_B64 = {
"cases_long.csv": "...",
"raw_population.csv": "H4sIAKs94moC/02QvXKDMBCEe57CHc0V+kEISmdSOJm4SfICp4FEYkB4FBEGP31OkouUe/vdak/f6+8Y/BowjnBbb9uM0a2+Os+nyzbgMaAFKWXHGEuzV9y/oFUyySsGZ0C2/OFd0WIgnLfNY/KU9kGrPmtzoAel/+G7i6BTFqs+0GONwBud5QWniaKELC7hzxbnsR6JLwELLcQTGecftyS0E7kkcQsGEIyLJN/xSG7LUm71ie5+B9H3GX1DOwGllXrD6KEX+bIXYwhSuhSrkY7omcrKotlzXlNqDoGe2yJ0ohRdAh3JefmUP5Am0edcAQAA",
"governorate_coords.csv": "H4sIAKs94moC/2VVwW4bRwy95yt802W64Aw55PDoNi3SIr40vfRkjKJtrUSWgpVc10Y/vo8ryZYRQNrZXUiPbx4f33z7vFuN6e/dP+O03U39MCa8uN32+zFtdtu06Yf07+3X+zSN33bTYX/7ue/HfdreTrvHfVo+bDbjYb2df79/9+fPmdP1fd/2w9X15up6v77vd+kTnhf96qf14SmJDIW1VeNWScVFUq6DsOasXtzViTjRQOmP6WFMmfU7PFQpfAI9L4Blwd/FrSlZbTVgmZs2VZTLXCzeWM5ixCrZhdhfixxxAtoSKt30u8f14eJOeKhC4JebSTXPM282aUWZa+UmVZLJ4IRypTleZcsXuzhDRQ3Hpqa+PV2B7VSqgplSzp6yDllVWBxqYFdmqbXBuHJFPasZBC71AUigUnp/1+/7dF4gSg7GGVo7lwpgGaq3xqyVoLk3T84DtqRWTIvWckn5CANkzun3/hStPC1gjH84S3MFqRrAWpsol6wlO7eUqQ5VK+So7oUyXTA+ooRbLH3oX74A8bQAGDyVc1GXAr1DCwpgga6tttC92EBRvBaVYgaQF+QjTCC3UPzDw6o/rYAb97D5enWskdVIgEfWAijoN6aa3bzUbAQVMuNn2Vu25qUVsfaml2fkq/9gzB/OT9EFTTd9Wi9xXcQikKFQa9IaGlirzn73Sm5ctBhlqMc2mMKVTWuFezEBL7UA06dxiTozbGwtp1+Xy/kbHdZGTCYGmwdtGcL+8LgZG5wZPs0YMFZgw5Swj7yiAyQQJbb0Y2zh9UZiflShdMOYwkWBzYUKhipng8Eb9mJ54Gqihn6xFX4j0gwU/qF4gp0242JM16ur932zXgR7U8LYSnWMvwUeD7CUFHYy6GFQPTdCZKANHPPNJPVNjRNqKF8wxYu+ilBYnBoN58E9IqVVAUPwhZswUAVex+SqQ/ymgwOckD+ZGms549cTXig01/qtP/71skZjPQYL8kQecFgV8SDxMUMRcHdMckZrKwrE6zfUA2b2z23cRRVsra+fn3FdrJ+DPwTIGrGDfChhykGywqEImGiJeCrFB0enDQ/oRLlo7owVwtT0sd99OV4gekVUoWVEpDa7b4C6rZQSGQbfpFLxypEIgl4AmPQVNECCaknXy6eIsPkq2GcMJ7I9Q8g6dxIz1qwRYg2m0ZowRkMlqpG/BJUuUGeUoAonrsbt8QKqjTKTwhoZWYP9F8wpTOw1UxHJmPLi6AMh8BFdsCFfSgyQwMzp011fPsZJdFpB1+EpZlDBnEgJbyMA0SEcTgLXIxmL2wCXS7DP4VG+MMYRKIRwhM5qQlI+HM53jzgvHIIwSEdis0vkFuwBDyB/MbFIAOMEuTE+VHBmcYTlsQDSMt3svm42He741J8Wu3kb7XSG9OkYaOdbjHdFs2DdOHoIWRWlrEabAQyZPCf4cYBTFWeJgI697CWn648BNZ2y7AwbY4uB2n3eHab+ssIYjWqMJ84gCBdpPCDa3OJwdOhkeAUqGDO021VxrDOnX/pmPyZK7/4HGgVU6W4IAAA="
}

In [ ]:
import base64, gzip, os, hashlib
os.makedirs("data_real", exist_ok=True)
for name, b64 in DATA_B64.items():
    raw = gzip.decompress(base64.b64decode(b64))
    with open(os.path.join("data_real", name), "wb") as f:
        f.write(raw)
    print(f"{name:28s} {len(raw):>9,d} bytes   sha256 {hashlib.sha256(raw).hexdigest()[:16]}")

In [ ]:
import pandas as pd
cases = pd.read_csv("data_real/cases_long.csv")
coords = pd.read_csv("data_real/governorate_coords.csv")
print("panel rows", len(cases), "| governorates", cases.governorate.nunique(),
      "| bulletin dates", cases.date.nunique())
print("elapsed days", cases.elapsed.min(), "to", cases.elapsed.max())
print("interval_days values:", sorted(cases.interval_days.dropna().unique().astype(int)))
print("exact-zero incidence rows:", int((cases.incidence_rate == 0).sum()))
display(coords[["pcode", "governorate", "lon", "lat", "x_km", "reports_cases"]].head(8))

### 1.1 Where this panel disagrees with the project's earlier `cases_long.csv`

Stated here rather than left for a reader to trip over. The panel rebuilt in this notebook
reproduces two of the three counts the methodology draft asserts and contradicts the third.

| quantity | this notebook | methodology draft | status |
|---|---|---|---|
| rows carrying a defined `incidence_rate` | 2,806 | 2,806 | agrees |
| `interval_days` support | $\{1,2,3,4,5,7,11,14,21\}$ | $\{1,2,3,4,5,7,11,14,21\}$ | agrees |
| exact-zero incidence rows | **276** | **622** | **disagrees** |

The row count agrees exactly: the draft's 2,827 raw rows less one un-differenced first bulletin per
governorate gives 2,806, which is what this panel holds. The zero count does not, and the gap is
not cosmetic — it moves the zero fraction from 22 % to 10 %, and the zero fraction is what decides
whether the $1/\sqrt{\text{count}}$ weighting drives the fit subcritical. The cell below recomputes
it from the embedded file so the number in this notebook is checkable at source; resolving which
preprocessing step produces 622 is open, and until it is, the Poisson-deviance objective used
throughout is the choice that does not depend on the answer.

In [ ]:
z = cases[cases.incidence_rate.notna()]
print(f"rows with a defined incidence_rate : {len(z):,d}")
print(f"exact zeros                        : {int((z.incidence_rate == 0).sum()):,d}"
      f"   ({100*(z.incidence_rate == 0).mean():.1f} %)")
print(f"negative increments                : {int((z.new_cases < 0).sum())}")
print(f"governorates whose cumulative ever falls : "
      f"{int((cases.groupby('governorate').cumulative.apply(lambda s: (s.diff() < 0).any())).sum())}")
print("\nzero rows by governorate:")
print(z.assign(is0=z.incidence_rate == 0).groupby("governorate").is0.sum()
       .sort_values(ascending=False).head(8).to_string())

## 2. The model, the solver and the two architectures

The modules are written out here so the notebook is the single source. `seip.py` carries the PDE,
the kernels and an exact spectral diffusion propagator; `data_io.py` the panel loader and the
weighting schemes; `fit.py` the deterministic arm; `pinn.py` both network architectures.

A note on the two architectures, since the comparison is the point of the notebook.

**DPINN** — two separate multilayer perceptrons: a *FieldNet* mapping $(t,a)$ or $(t,b)$ to the
states, and a *BetaNet* mapping $t$ to $(\beta_1,\beta_2)$. They are coupled only through the
renewal residual. Both are plain $\tanh$ networks in the raw coordinate.

**FF-PINN** — the same state network, but each input coordinate is first lifted through a
deterministic Fourier feature embedding $\gamma(t)=[\sin(2\pi\omega_k t),\cos(2\pi\omega_k t)]_k$,
and $\beta$ is a truncated Fourier series with a linear trend rather than a free network. The
truncation order is bounded above by the record's own cadence: with a maximum reporting gap of
$\Delta_{\max}$ days over a window of $T$ days, orders beyond $M = T/(2\Delta_{\max})$ are not
resolved by the data, and the code refuses to exceed that ceiling.

In [ ]:
%%writefile seip.py
"""
seip.py -- Age-structured SEIP transport-diffusion model for the Yemen
second-wave cholera record.

    dS/dt            = D1 S_xx - S[int b1(t,a) I da + int b2(t,b) P db] + Lam - mu1 S
    dE/dt + dE/da    = D2 E_xx - (mu2 + sigma(a)) E
    dI/dt + dI/da    = D3 I_xx - (mu3 + gamma + delta) I
    dP/dt + dP/db    = D4 P_xx - mu4 P

    E(t,0,x) = S(t,x) [ int b1 I da + int b2 P db ]      <- the unknowns enter HERE
    I(t,0,x) = int sigma(a) E(t,a,x) da
    P(t,0,x) = int alpha(a) I(t,a,x) da

Homogeneous Neumann in x at both ends; no terminal condition in a or b -- the age
derivatives are outward transport, so a cohort reaching the truncation age leaves
the tracked population.

NUMERICS.  da = db = dt, so age advection is the exact characteristic shift and
introduces no numerical diffusion in age.  The time integrator is Strang
splitting about the reaction-transport operator,

    exp(dt/2 A) exp(dt B) exp(dt/2 A),      A = diffusion,  B = reaction+transport,

with the renewal inflows evaluated at the midpoint by a predictor-corrector, so
the scheme is second order in dt.  A first-order Lie variant is retained under
`scheme="lie"` because the earlier results in this project were produced with it
and a like-for-like comparison needs it.
"""

from __future__ import annotations

import numpy as np
from dataclasses import dataclass, field

# --------------------------------------------------------------------------
# Fixed constants. None of these are ever fitted.
# --------------------------------------------------------------------------
MU1 = MU2 = MU3 = 1.0 / (70.0 * 365.0)   # background mortality, /day
LAMBDA_REC = MU1                          # recruitment, population fraction /day
SIGMA0 = 1.0 / 1.4                        # latency progression, /day
GAMMA = 1.0 / 7.0                         # recovery, /day
DELTA = 0.001                             # disease-induced mortality, /day
MU4 = 1.0 / 30.0                          # environmental pathogen decay, /day
ALPHA0 = 2.0                              # shedding amplitude
A_MAX = 7.0                               # infection-age truncation, days
B_MAX = 30.0                              # pathogen-age truncation, days
X_MAX_FULL = 1051.806                     # full domain, km (Socotra)
X_MAX_OBSERVED = 795.396633               # farthest REPORTING governorate (Al Maharah)
T_MAX = 272.0                             # window, days (22 May 2017 - 18 Feb 2018)

R_I = MU3 + GAMMA + DELTA                 # infectious removal rate, /day
R_E = MU2 + SIGMA0                        # exposed removal rate, /day

# Residence times.  TRUNC is the occupancy the model actually holds at constant
# incidence; NAIVE is what the untruncated residence-time identity assumes.
#
# These ratios are NOT both 1/(1-e^-1).  mu4*B_MAX = 1 exactly, so the
# environmental ratio is 1.581977.  R_I*A_MAX = 1.0072740, because gamma = 1/7 is
# joined by delta and mu3, so the infectious ratio is 1.575332.  Earlier project
# text gave 1.58 for both and "about 58% more mass"; the infectious figure is
# 57.5%.
TAU_I_TRUNC = (1.0 - np.exp(-R_I * A_MAX)) / R_I      # 4.411419 d
TAU_I_NAIVE = 1.0 / R_I                               # 6.949450 d
TAU_P_TRUNC = (1.0 - np.exp(-MU4 * B_MAX)) / MU4      # 18.963617 d
TAU_P_NAIVE = 1.0 / MU4                               # 30.000000 d
RATIO_I = TAU_I_NAIVE / TAU_I_TRUNC                   # 1.575332
RATIO_P = TAU_P_NAIVE / TAU_P_TRUNC                   # 1.581977

# Exact kernel integrals (closed form, cross-checked against adaptive quadrature).
# int_0^7 a(7-a) e^{-a/12} da.  An earlier version of this file quoted 43.012,
# which was this module's own trapezoid value at dt=0.25, not the integral.
INT_F_DA = 43.0688907175
INT_H_DB = 1078.5


def f_kernel(a):
    """Infection-age transmission/shedding shape. Zero at 0 and A_MAX, peak at a=3."""
    a = np.asarray(a, dtype=float)
    return a * (A_MAX - a) * np.exp(-a / 12.0)


def h_kernel(b):
    """Pathogen-age infectivity: 700 on [0,1], 1399-699b on (1,2], 1 on (2,30]."""
    b = np.asarray(b, dtype=float)
    out = np.ones_like(b)
    out = np.where(b <= 1.0, 700.0, out)
    mid = (b > 1.0) & (b <= 2.0)
    out = np.where(mid, 1399.0 - 699.0 * b, out)
    return out


# --------------------------------------------------------------------------
# Grid
# --------------------------------------------------------------------------
@dataclass
class Grid:
    """da = db = dt is required, not a convenience: it makes the age shift exact.

    Quadrature in a and b is composite trapezoid on this same grid, which is a
    documented substitution for the Gauss-Legendre rule the methodology specifies.
    It is benign here -- at dt <= 0.25 both breakpoints of h land on grid nodes and
    h is continuous at both -- and the convergence of the next-generation
    integrals is reported by ngm_integrals(..., report=True).
    """
    dt: float = 0.25
    nx: int = 41
    x_max: float = X_MAX_FULL

    na: int = field(init=False); nb: int = field(init=False); nt: int = field(init=False)
    a: np.ndarray = field(init=False); b: np.ndarray = field(init=False)
    x: np.ndarray = field(init=False); t: np.ndarray = field(init=False)
    dx: float = field(init=False)
    wa: np.ndarray = field(init=False); wb: np.ndarray = field(init=False)
    wf: np.ndarray = field(init=False); wh: np.ndarray = field(init=False)
    wsig: np.ndarray = field(init=False); walpha: np.ndarray = field(init=False)

    def __post_init__(self):
        for name, L in (("A_MAX", A_MAX), ("B_MAX", B_MAX), ("T_MAX", T_MAX)):
            n = L / self.dt
            if abs(n - round(n)) > 1e-9:
                raise ValueError(f"dt={self.dt} does not divide {name}={L}; "
                                 "da=db=dt is required for the exact age shift")
        self.na = int(round(A_MAX / self.dt))
        self.nb = int(round(B_MAX / self.dt))
        self.nt = int(round(T_MAX / self.dt))
        self.a = np.linspace(0.0, A_MAX, self.na + 1)
        self.b = np.linspace(0.0, B_MAX, self.nb + 1)
        self.t = np.linspace(0.0, T_MAX, self.nt + 1)
        if self.nx == 1:
            # Single location: the spatial operator is switched off entirely, so
            # dx is never used. Only legal with D1=..=D4=0.
            self.x = np.array([0.0]); self.dx = np.inf
        else:
            self.x = np.linspace(0.0, self.x_max, self.nx)
            self.dx = self.x[1] - self.x[0]

        self.wa = np.full(self.na + 1, self.dt); self.wa[0] *= 0.5; self.wa[-1] *= 0.5
        self.wb = np.full(self.nb + 1, self.dt); self.wb[0] *= 0.5; self.wb[-1] *= 0.5
        self.wf = self.wa * f_kernel(self.a)
        self.wh = self.wb * h_kernel(self.b)
        self.wsig = self.wa * SIGMA0
        self.walpha = self.wa * ALPHA0 * f_kernel(self.a)

    def diffusion_number(self, diff):
        """D dt / dx^2, reported for orientation only. The diffusion sub-flow is
        applied by its exact spectral propagator, so there is NO CFL restriction
        and this number does not bound the step."""
        if self.nx == 1:
            return 0.0
        return max(diff.as_tuple()) * self.dt / self.dx ** 2


def ngm_integrals(grid: Grid = None, report=False):
    """kappa_EI, J1, J2.  Published: 0.993209, 27.477214, 1040.759087.

    With grid=None these are computed by adaptive quadrature and are exact to
    ~1e-10. Passing a Grid returns that grid's trapezoid values instead, which is
    what a fit on that grid actually uses.
    """
    if grid is None:
        from scipy.integrate import quad
        kw = dict(epsabs=1e-12, epsrel=1e-12, limit=200)
        kappa = quad(lambda a: SIGMA0 * np.exp(-R_E * a), 0, A_MAX, **kw)[0]
        j1 = quad(lambda a: f_kernel(a) * np.exp(-R_I * a), 0, A_MAX, **kw)[0]
        j2 = sum(quad(lambda b: h_kernel(b) * np.exp(-MU4 * b), lo, hi, **kw)[0]
                 for lo, hi in ((0, 1), (1, 2), (2, B_MAX)))
    else:
        kappa = float(np.sum(grid.wa * SIGMA0 * np.exp(-R_E * grid.a)))
        j1 = float(np.sum(grid.wa * f_kernel(grid.a) * np.exp(-R_I * grid.a)))
        j2 = float(np.sum(grid.wb * h_kernel(grid.b) * np.exp(-MU4 * grid.b)))
    if report:
        pub = (0.993209, 27.477214, 1040.759087)
        for nm, got, want in zip(("kappa_EI", "J1", "J2"), (kappa, j1, j2), pub):
            print(f"  {nm:<9s} {got:>14.6f}  published {want:>12.6f}  "
                  f"rel {abs(got-want)/want:.2e}")
    return kappa, j1, j2


# --------------------------------------------------------------------------
# Diffusion
# --------------------------------------------------------------------------
def laplacian_x(u, dx, stencil="fv"):
    """Second difference along the LAST axis with zero-flux ends.

    stencil='fv'  finite-volume / cell-centred: no flux across the two outer
                  faces, so the end rows are (u1-u0) and (u_{N-2}-u_{N-1}).
                  SYMMETRIC and MASS-CONSERVING (columns sum to zero), and
                  diagonalised exactly by the DCT-II.
    stencil='ghost'  node-centred mirror, end rows 2(u1-u0). This is what earlier
                  versions of this project used. It is neither symmetric nor
                  conservative -- total mass drifts under pure diffusion -- and is
                  kept only so an earlier result can be reproduced on request.
    """
    lap = np.empty_like(u)
    lap[..., 1:-1] = u[..., 2:] - 2.0 * u[..., 1:-1] + u[..., :-2]
    c = 1.0 if stencil == "fv" else 2.0
    lap[..., 0] = c * (u[..., 1] - u[..., 0])
    lap[..., -1] = c * (u[..., -2] - u[..., -1])
    return lap / (dx * dx)


def diffusion_eigenvalues(nx, dx):
    """Eigenvalues of the finite-volume Neumann Laplacian, whose eigenvectors are
    the DCT-II basis: lambda_j = -4 sin^2(pi j / 2N) / dx^2."""
    j = np.arange(nx)
    return -4.0 * np.sin(np.pi * j / (2.0 * nx)) ** 2 / (dx * dx)


def diffuse_exact(u, factor, axis=-1):
    """Apply exp(tau D L) exactly, where `factor` = exp(tau D lambda_j).

    Spectral, so this is the exact solution operator of the semi-discrete
    diffusion problem rather than an approximation of it: there is no splitting
    error from this sub-flow and no CFL restriction at all. Mass is conserved to
    machine precision because lambda_0 = 0.
    """
    from scipy.fft import dct, idct
    return idct(factor * dct(u, type=2, norm="ortho", axis=axis),
                type=2, norm="ortho", axis=axis)


@dataclass
class Diffusion:
    D1: float = 50.0
    D2: float = 50.0
    D3: float = 50.0
    D4: float = 50.0

    def as_tuple(self):
        return (self.D1, self.D2, self.D3, self.D4)

    def label(self):
        v = self.as_tuple()
        return f"D={v[0]:g}" if len(set(v)) == 1 else \
               "D=(" + ",".join(f"{z:g}" for z in v) + ")"

    def length_scale(self, T=T_MAX):
        """sqrt(2 D T): how far mass spreads over the window, per field."""
        return tuple(np.sqrt(2.0 * D * T) for D in self.as_tuple())


# --------------------------------------------------------------------------
# Initial condition
# --------------------------------------------------------------------------
@dataclass
class InitialCondition:
    S0: np.ndarray
    I0: np.ndarray
    P0: np.ndarray
    E0: np.ndarray
    I_agg_nodal: np.ndarray
    P_agg_nodal: np.ndarray
    S_nodal: np.ndarray
    opts: dict = field(default_factory=dict)


def _lognormal_profile(v, mean, sd, w):
    """Log-normal density on the grid v, moment-matched to (mean, sd), truncated
    to the grid and renormalized to unit mass under the weights w."""
    s2 = np.log(1.0 + (sd / mean) ** 2)
    mu = np.log(mean) - 0.5 * s2
    s = np.sqrt(s2)
    g = np.zeros_like(v)
    pos = v > 0
    g[pos] = np.exp(-(np.log(v[pos]) - mu) ** 2 / (2 * s2)) / (v[pos] * s * np.sqrt(2 * np.pi))
    tot = float(np.sum(w * g))
    return g / tot if tot > 0 else g


def build_initial_condition(grid: Grid, xg, y_first, cum_first,
                            truncation_correction: bool = False,
                            age_profile: str = "flat",
                            p_from_model: bool = False) -> InitialCondition:
    """Data-anchored initial state.

    truncation_correction=False reproduces the project identity
        I_agg0 = y0 / (mu3+gamma+delta),  P_agg0 = alpha0 I_agg0 / mu4,
    which inverts an UNtruncated mean duration. True uses the occupancy the
    truncated system actually holds -- smaller by 1.5753 (I) and 1.5820 (P).

    age_profile: 'flat' (project baseline, adds no assumption the synthetic IC did
    not already carry), 'equilibrium' (exponential, consistent with the stock
    identity), or 'lognormal' (mean/SD 3.5/2.0 d for I and 10/6 d for P, the
    profile the methodology records as the available refinement).
    """
    tau_I = TAU_I_TRUNC if truncation_correction else TAU_I_NAIVE
    tau_P = TAU_P_TRUNC if truncation_correction else TAU_P_NAIVE

    y_first = np.asarray(y_first, float)
    I_agg = y_first * tau_I
    P_agg = ALPHA0 * I_agg * tau_P
    R_frac = np.maximum(0.0, np.asarray(cum_first, float) - I_agg)
    S_nodal = np.maximum(0.0, 1.0 - I_agg - R_frac)

    order = np.argsort(xg)
    xs = np.asarray(xg, float)[order]

    def lift(vals):
        """Nodal governorate values -> field on the x grid: piecewise linear
        between reporting coordinates, held flat beyond the farthest one."""
        v = np.asarray(vals, float)[order]
        if grid.nx == 1:
            return np.array([v[0]])
        return np.interp(grid.x, xs, v, left=v[0], right=v[-1])

    S0 = lift(S_nodal); I_agg_f = lift(I_agg); P_agg_f = lift(P_agg)

    if age_profile == "flat":
        pI = np.ones_like(grid.a) / A_MAX
        pP = np.ones_like(grid.b) / B_MAX
    elif age_profile == "equilibrium":
        pI = np.exp(-R_I * grid.a); pI /= np.sum(grid.wa * pI)
        pP = np.exp(-MU4 * grid.b); pP /= np.sum(grid.wb * pP)
    elif age_profile == "lognormal":
        pI = _lognormal_profile(grid.a, 3.5, 2.0, grid.wa)
        pP = _lognormal_profile(grid.b, 10.0, 6.0, grid.wb)
    else:
        raise ValueError(f"unknown age_profile {age_profile!r}")

    I0 = pI[:, None] * I_agg_f[None, :]
    P0 = pP[:, None] * P_agg_f[None, :]

    if p_from_model:
        # The project IC sets P_agg0 = alpha0 I_agg0 / mu4, treating the shedding
        # inflow as alpha0 * I_agg. The model's own renewal condition is
        # alpha0 * int f(a) I da, larger by int f da / A_MAX = 6.1527 under a flat
        # profile. This rescales P(0,.) to the stock that renewal implies, so the
        # IC and the equation it feeds agree at t=0.
        shed = grid.walpha @ I0
        cur = grid.wb @ P0
        scale = np.divide(shed * tau_P, cur, out=np.ones_like(cur), where=cur > 0)
        P0 = P0 * scale[None, :]
        P_agg = P_agg * (scale[0] if grid.nx == 1 else
                         np.interp(np.asarray(xg, float), grid.x, scale))

    E0 = np.zeros((grid.na + 1, grid.nx))
    return InitialCondition(S0, I0, P0, E0, I_agg, P_agg, S_nodal,
                            opts=dict(truncation_correction=truncation_correction,
                                      age_profile=age_profile,
                                      p_from_model=p_from_model,
                                      tau_I=tau_I, tau_P=tau_P))


# --------------------------------------------------------------------------
# Forward solve
# --------------------------------------------------------------------------
OBS_FORCE_OF_INFECTION = "foi"      # H = S [int b1 I da + int b2 P db]
OBS_INFECTIOUS_ONSET = "onset"      # H = int sigma(a) E da, the declared alternative


def solve_forward(grid: Grid, ic: InitialCondition, beta10_t, beta20_t,
                  diff: Diffusion, want_states: bool = False,
                  scheme: str = "strang", observable: str = OBS_FORCE_OF_INFECTION,
                  n_sweeps: int = 2, record_full: bool = False):
    """March over [0, T_MAX].

    scheme
        'strang'  Strang splitting about the reaction-transport flow, with the
                  diffusion half-flows applied EXACTLY (spectral) and the renewal
                  boundary values taken at the NEW time level by a predictor-
                  corrector. Second order in dt.
        'lie'     first-order Lie splitting with the renewal inflows held at the
                  old time level: what this project's earlier results used.

    observable
        'foi'     S [int b1 I da + int b2 P db], the baseline.
        'onset'   int sigma(a) E da, the declared sensitivity alternative.

    Returns H (nt+1, nx), Sbar (nt+1,), and optionally recorded states.
    """
    if scheme not in ("strang", "lie"):
        raise ValueError(f"unknown scheme {scheme!r}")
    if observable not in (OBS_FORCE_OF_INFECTION, OBS_INFECTIOUS_ONSET):
        raise ValueError(f"unknown observable {observable!r}")
    dt = grid.dt
    Ds = diff.as_tuple()
    if grid.nx == 1 and any(Ds):
        raise ValueError("a single-location grid carries no spatial operator; "
                         "nx=1 requires D1=D2=D3=D4=0")

    S = ic.S0.copy(); E = ic.E0.copy(); I = ic.I0.copy(); P = ic.P0.copy()
    decE = np.exp(-R_E * dt); decI = np.exp(-R_I * dt); decP = np.exp(-MU4 * dt)

    # Exact diffusion propagators. Strang applies two half-steps per step; Lie one
    # full step. Precomputed once -- they do not depend on t.
    spatial = grid.nx > 1 and any(Ds)
    if spatial:
        eig = diffusion_eigenvalues(grid.nx, grid.dx)
        tau = 0.5 * dt if scheme == "strang" else dt
        fac = [np.exp(D * tau * eig) if D else None for D in Ds]

    def diffuse(S, E, I, P):
        if not spatial:
            return S, E, I, P
        if fac[0] is not None: S = diffuse_exact(S, fac[0])
        if fac[1] is not None: E = diffuse_exact(E, fac[1])
        if fac[2] is not None: I = diffuse_exact(I, fac[2])
        if fac[3] is not None: P = diffuse_exact(P, fac[3])
        return S, E, I, P

    H = np.empty((grid.nt + 1, grid.nx))
    Sbar = np.empty(grid.nt + 1)
    want_states = want_states or record_full
    rec = {k: np.empty((grid.nt + 1, grid.nx))
           for k in ("I_agg", "P_agg", "E_agg", "foi")} if want_states else None
    if record_full:
        # The full age-resolved fields. Only sane at small nx -- for a single
        # location this is (nt+1, na+1), which is what a city-level I(t,a)
        # surface figure needs.
        rec["I_full"] = np.empty((grid.nt + 1, grid.na + 1, grid.nx))
        rec["E_full"] = np.empty((grid.nt + 1, grid.na + 1, grid.nx))
        rec["P_full"] = np.empty((grid.nt + 1, grid.nb + 1, grid.nx))

    for k in range(grid.nt + 1):
        lamI = beta10_t[k] * (grid.wf @ I)
        lamP = beta20_t[k] * (grid.wh @ P)
        foi0 = lamI + lamP
        H[k] = S * foi0 if observable == OBS_FORCE_OF_INFECTION else (grid.wsig @ E)
        Sbar[k] = S.mean()
        if want_states:
            rec["I_agg"][k] = grid.wa @ I
            rec["P_agg"][k] = grid.wb @ P
            rec["E_agg"][k] = grid.wa @ E
            rec["foi"][k] = foi0
            if record_full:
                rec["I_full"][k] = I; rec["E_full"][k] = E; rec["P_full"][k] = P

        if k == grid.nt:
            break

        if scheme == "strang":
            S, E, I, P = diffuse(S, E, I, P)
            foi0 = (beta10_t[k] * (grid.wf @ I)
                    + beta20_t[k] * (grid.wh @ P))          # after the half-flow

        inE0 = S * foi0
        inI0 = grid.wsig @ E
        inP0 = grid.walpha @ I

        # age/duration transport: exact characteristic shift plus exact decay
        En = np.empty_like(E); In = np.empty_like(I); Pn = np.empty_like(P)
        En[1:] = E[:-1] * decE; In[1:] = I[:-1] * decI; Pn[1:] = P[:-1] * decP
        En[0] = inE0; In[0] = inI0; Pn[0] = inP0            # predictor

        if scheme == "lie":
            S = S + dt * (-S * foi0 + LAMBDA_REC - MU1 * S)
            np.clip(S, 0.0, None, out=S)
        else:
            b10h, b20h = beta10_t[k + 1], beta20_t[k + 1]
            Snew = S
            for _ in range(max(1, n_sweeps)):
                foi1 = b10h * (grid.wf @ In) + b20h * (grid.wh @ Pn)
                # S by the trapezoid rule, solved semi-implicitly so the sink uses
                # the new-time S rather than the old one
                Snew = ((S * (1.0 - 0.5 * dt * (foi0 + MU1)) + dt * LAMBDA_REC)
                        / (1.0 + 0.5 * dt * (foi1 + MU1)))
                np.clip(Snew, 0.0, None, out=Snew)
                # The renewal conditions are constraints at the NEW time level, so
                # the a=0 / b=0 cells take their new-time values, not a time
                # average of old and new. Swept in causal order: E(.,0) needs I,P;
                # I(.,0) needs E; P(.,0) needs I.
                En[0] = Snew * foi1
                In[0] = grid.wsig @ En
                Pn[0] = grid.walpha @ In
            S = Snew
        E, I, P = En, In, Pn

        np.clip(E, 0.0, None, out=E)
        np.clip(I, 0.0, None, out=I)
        np.clip(P, 0.0, None, out=P)

        S, E, I, P = diffuse(S, E, I, P)

    if want_states:
        rec.update(S_final=S, I_final=I, P_final=P)
        return H, Sbar, rec
    return H, Sbar, None


def R0_of_t(beta10_t, beta20_t, Sbar, kappa, J1, J2):
    """R0(t) = Sbar(t) kappa_EI J1 (beta10 + alpha0 beta20 J2).

    Always call this rather than re-deriving the expression inline: reusing the
    formula or its constituent integrals across reductions without rederiving is
    a documented failure mode in this project's earlier work, and dropping Sbar
    silently compares two different definitions.
    """
    return np.asarray(Sbar) * kappa * J1 * (np.asarray(beta10_t)
                                            + ALPHA0 * np.asarray(beta20_t) * J2)


def route_share(beta10_t, beta20_t, J2, grid=None):
    """Environmental share of the next-generation contribution, in [0, 1]:

        alpha0 beta20 J2 / (beta10 + alpha0 beta20 J2)

    Reported in preference to the ratio alpha0 beta20 J2 / beta10, whose POINTWISE
    mean is dominated by the stretches where beta10 sits on the positivity floor
    and is therefore an artifact of that floor rather than a statement about the
    routes. The time-integrated ratio is returned alongside for continuity with
    the earlier column.
    """
    b10 = np.asarray(beta10_t, float); b20 = np.asarray(beta20_t, float)
    env = ALPHA0 * b20 * J2
    share_t = env / np.maximum(b10 + env, 1e-300)
    if grid is None:
        share = float(np.mean(share_t))
        ratio = float(np.sum(env) / max(np.sum(b10), 1e-300))
    else:
        w = np.full(grid.t.size, grid.dt); w[0] *= 0.5; w[-1] *= 0.5
        share = float(np.sum(w * share_t) / np.sum(w))
        ratio = float(np.sum(w * env) / max(np.sum(w * b10), 1e-300))
    return share, ratio, share_t

In [ ]:
%%writefile data_io.py
"""
data_io.py -- Load, clean and winsorize the Yemen governorate bulletin panel,
build the observation design, and generate a synthetic twin that matches the
record's SAMPLING DESIGN rather than only its size.

Expects the project's own files in --data-dir:
    cases_long.csv         cleaned, name-harmonized, spatially-referenced panel
    governorate_coords.csv governorate -> great-circle distance from Sana'a (km)
    raw_population.csv     governorate -> population

Column names differ between exports, so COLUMN_HINTS does fuzzy detection and
the loader REPORTS what it matched. Override via COLUMN_OVERRIDE.
"""

from __future__ import annotations

import os
import warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass, field

import seip

COLUMN_OVERRIDE = {
    "governorate": None, "date": None, "cumulative": None, "new_cases": None,
    "incidence_rate": None, "interval_days": None, "x_km": None, "population": None,
}

COLUMN_HINTS = {
    "governorate": ["governorate", "gov", "admin1", "region", "name"],
    "date": ["date", "report_date", "day"],
    "cumulative": ["cum", "cumulative", "total_cases", "cases_cum"],
    "new_cases": ["new_cases", "new", "delta_cases", "d_cases", "cases_new", "increment"],
    "incidence_rate": ["incidence_rate", "percapita_incidence", "incidence", "per_capita"],
    "interval_days": ["interval_days", "interval", "dt_days", "gap"],
    "x_km": ["distance", "x_km", "dist_km", "x", "km"],
    "population": ["population", "pop", "n_g", "total"],
}

WINSOR_Q = 0.995
# The record's window opens here (problem statement Sec. 1). Used so that a
# cases_long.csv carrying the first wave or the 2018 resurgence does not silently
# shift the time origin.
WINDOW_START = "2017-05-22"


def _match(cols, key):
    if COLUMN_OVERRIDE.get(key):
        return COLUMN_OVERRIDE[key]
    low = {c.lower().replace(" ", "_"): c for c in cols}
    for hint in COLUMN_HINTS[key]:
        for lc, orig in low.items():
            if lc == hint:
                return orig
    for hint in COLUMN_HINTS[key]:
        for lc, orig in low.items():
            if hint in lc:
                return orig
    return None


@dataclass
class Panel:
    """The fitting design. One row per (governorate, bulletin interval)."""
    gov: np.ndarray
    t0: np.ndarray
    t1: np.ndarray
    y: np.ndarray                # per-capita incidence after winsorization
    y_raw: np.ndarray
    dC: np.ndarray               # raw case increment after winsorization
    dC_raw: np.ndarray
    w: np.ndarray                # weight in force
    w_count: np.ndarray          # 1/sqrt(max(dC,1))        -- as specified
    w_rate: np.ndarray           # N dt / sqrt(max(dC,1))   -- rate-consistent
    names: list
    xg: np.ndarray
    pop: np.ndarray
    y_first: np.ndarray
    cum_first: np.ndarray
    t_first: np.ndarray
    winsor_cut: float
    n_winsorized: int
    true_zero: np.ndarray = None      # (ng,) structural zero seed
    first_hold: np.ndarray = None     # (ng,) first bulletin was a reporting hold
    provenance: dict = field(default_factory=dict)
    synthetic: bool = False
    weight_kind: str = "count"

    @property
    def ng(self):
        return len(self.names)

    @property
    def n_obs(self):
        return len(self.y)

    def set_weights(self, kind="count", yhat=None):
        """'count'  1/sqrt(max(dC,1)) -- the weight the methodology specifies.
        'rate'   N dt / sqrt(max(dC,1)) -- what a per-capita RATE residual calls
                 for, since Var(y) = dC/(N dt)^2 for a Poisson count.
        'model'  1/sqrt(max(yhat N dt,1)) scaled to a rate -- iteratively
                 reweighted, using the FITTED mean rather than the realization.

        'count' and 'rate' both put the observed count in the denominator, so a
        row that happened to fluctuate low is up-weighted and an observed zero is
        up-weighted maximally. That is the classic downward bias of
        Poisson-weighted least squares and it acts in the SAME direction as the
        initial-condition truncation mismatch. 'model' removes it: see
        fit.fit(..., irls=True).
        """
        if kind == "count":
            w = self.w_count
        elif kind == "rate":
            w = self.w_rate
        elif kind == "none":
            w = np.ones_like(self.w_count)
        elif kind == "model":
            if yhat is None:
                raise ValueError("kind='model' needs yhat")
            Ndt = self.pop[self.gov] * (self.t1 - self.t0)
            w = Ndt / np.sqrt(np.maximum(np.asarray(yhat, float) * Ndt, 1.0))
        else:
            raise ValueError(kind)
        self.w = w / w.mean()
        self.weight_kind = kind
        return self

    def zero_weight_share(self):
        """Fraction of total squared weight carried by exact-zero rows. On the
        record this is the dominant part of the objective; a synthetic twin that
        under-represents it does not test the thing being relied on."""
        z = self.y_raw == 0
        return float(np.sum(self.w[z] ** 2) / np.sum(self.w ** 2))

    def describe(self):
        iv = self.t1 - self.t0
        days = np.unique(np.round(np.concatenate([self.t0, self.t1]), 6))
        return (f"{self.n_obs} fitting rows, {self.ng} governorates, "
                f"{len(days)} distinct bulletin days, "
                f"{int((self.y_raw == 0).sum())} exact zeros "
                f"({100*np.mean(self.y_raw == 0):.1f}%), "
                f"interval days {iv.min():.3g}-{iv.max():.3g}, "
                f"winsor {self.winsor_cut:.4g} ({self.n_winsorized} rows), "
                f"zero-row weight share {100*self.zero_weight_share():.1f}%")


def load_panel(data_dir="data", winsorize=True, t_max=seip.T_MAX,
               window_start=WINDOW_START, verbose=True) -> Panel:
    cases_p = os.path.join(data_dir, "cases_long.csv")
    coords_p = os.path.join(data_dir, "governorate_coords.csv")
    pop_p = os.path.join(data_dir, "raw_population.csv")
    if not os.path.exists(cases_p):
        raise FileNotFoundError(
            f"{cases_p} not found. Put cases_long.csv, governorate_coords.csv and "
            f"raw_population.csv in {data_dir}/, or run with --synthetic.")

    df = pd.read_csv(cases_p)
    c = {k: _match(df.columns, k) for k in COLUMN_HINTS}
    if verbose:
        print("cases_long.csv columns matched:", {k: v for k, v in c.items() if v})
    if not c["governorate"] or not c["date"]:
        raise ValueError(f"no governorate/date column among {list(df.columns)}; "
                         "set COLUMN_OVERRIDE in data_io.py")
    df = df.rename(columns={v: k for k, v in c.items() if v})
    df["governorate"] = df["governorate"].astype(str).str.strip()
    df["date"] = pd.to_datetime(df["date"])

    if os.path.exists(coords_p):
        cd = pd.read_csv(coords_p)
        cd = cd.rename(columns={_match(cd.columns, "governorate"): "governorate",
                                _match(cd.columns, "x_km"): "x_km"})
        cd["governorate"] = cd["governorate"].astype(str).str.strip()
        coords = dict(zip(cd["governorate"], cd["x_km"].astype(float)))
    elif "x_km" in df.columns:
        coords = df.groupby("governorate")["x_km"].first().to_dict()
    else:
        raise ValueError("no governorate_coords.csv and no distance column")

    if "population" in df.columns:
        pops = df.groupby("governorate")["population"].first().to_dict()
    elif os.path.exists(pop_p):
        pp = pd.read_csv(pop_p)
        pp = pp.rename(columns={_match(pp.columns, "governorate"): "governorate",
                                _match(pp.columns, "population"): "population"})
        pp["governorate"] = pp["governorate"].astype(str).str.strip()
        pops = dict(zip(pp["governorate"], pp["population"].astype(float)))
    else:
        raise ValueError("no population column and no raw_population.csv")

    prov = {}
    missing = sorted(set(df["governorate"]) - set(coords) | set(df["governorate"]) - set(pops))
    if missing:
        prov["dropped_governorates"] = missing
        if verbose:
            print(f"  dropped {len(missing)} governorate(s) lacking a coordinate or "
                  f"population: {', '.join(missing)}")
    df = df[df["governorate"].isin(coords) & df["governorate"].isin(pops)].copy()

    # Explicit time origin. df.date.min() would silently shift the window if the
    # file carries the first wave or the 2018 resurgence.
    origin = pd.Timestamp(window_start) if window_start else df["date"].min()
    prov["window_start"] = str(origin.date())
    n_before = len(df)
    df = df[(df["date"] >= origin) &
            (df["date"] <= origin + pd.Timedelta(days=t_max))]
    prov["rows_outside_window"] = int(n_before - len(df))
    if prov["rows_outside_window"] and verbose:
        print(f"  {prov['rows_outside_window']} row(s) outside the "
              f"{t_max:g}-day window from {prov['window_start']} dropped")
    df["elapsed"] = (df["date"] - origin).dt.days.astype(float)
    df = df.sort_values(["governorate", "elapsed"])
    # Each row of an increment panel is a whole interval ending at `elapsed`.
    # _assemble differences consecutive rows, so without an opening snapshot it
    # treats the first row as a baseline and silently drops that governorate's
    # first interval (21 intervals, 16,598 cases on the record). Prepend the
    # snapshot at the start of the first interval so every row is fitted.
    if {"cumulative", "new_cases", "interval_days"} <= set(df.columns):
        first = df.groupby("governorate", as_index=False).head(1).copy()
        first["elapsed"] = first["elapsed"] - first["interval_days"]
        first["date"] = first["date"] - pd.to_timedelta(first["interval_days"], unit="D")
        first["cumulative"] = first["cumulative"] - first["new_cases"]
        first[["new_cases", "incidence_rate"]] = np.nan
        df = (pd.concat([first, df], ignore_index=True)
                .sort_values(["governorate", "elapsed"], kind="stable"))
    return _assemble(df, coords, pops, winsorize, False, prov, verbose)


def _assemble(df, coords, pops, winsorize, synthetic, prov=None, verbose=False):
    prov = dict(prov or {})
    names = sorted(df["governorate"].unique(), key=lambda g: coords[g])
    gidx = {g: i for i, g in enumerate(names)}
    xg = np.array([coords[g] for g in names], float)
    pop = np.array([pops[g] for g in names], float)

    rows = []
    y_first = np.zeros(len(names)); cum_first = np.zeros(len(names))
    t_first = np.zeros(len(names))
    first_hold = np.zeros(len(names), bool); true_zero = np.zeros(len(names), bool)
    n_dupe = n_monotone = 0
    have_cum = "cumulative" in df.columns

    for g, sub in df.groupby("governorate"):
        i = gidx[g]; N = pops[g]
        sub = sub.sort_values("elapsed")
        # Merge duplicate-date rows instead of dropping their case mass. For a
        # cumulative series the last report on a date wins; for increments they sum.
        if sub["elapsed"].duplicated().any():
            n_dupe += int(sub["elapsed"].duplicated().sum())
            agg = {"cumulative": "max"} if have_cum else {"new_cases": "sum"}
            keep = [c for c in sub.columns if c in agg]
            sub = sub.groupby("elapsed", as_index=False).agg({k: agg[k] for k in keep})
            sub = sub.sort_values("elapsed")
        el = sub["elapsed"].to_numpy(float)
        if have_cum:
            cum = sub["cumulative"].to_numpy(float)
            fixed = np.maximum.accumulate(cum)
            n_monotone += int((fixed != cum).sum())
            cum = fixed
            dC = np.diff(cum)
        elif "new_cases" in sub.columns:
            nc = sub["new_cases"].to_numpy(float)
            dC = nc[1:]
            # cum_first must be the cumulative burden BEFORE the first interval.
            # Rebuilding it as a cumsum from zero would force R(0)=0 everywhere.
            cum = np.concatenate([[np.nan], np.cumsum(nc[1:])])
        else:
            raise ValueError("need a cumulative or new-case column")
        dt_ = np.diff(el)
        keep = dt_ > 0
        if "incidence_rate" in sub.columns and len(sub) == len(el):
            y = sub["incidence_rate"].to_numpy(float)[1:]
        else:
            y = np.where(keep, dC / (N * np.where(keep, dt_, 1.0)), 0.0)
        for k in np.flatnonzero(keep):
            rows.append((i, el[k], el[k + 1], y[k], max(dC[k], 0.0)))
        # The IC anchor must be the first interval that actually OBSERVES
        # transmission. Taking literally the first interval conflates two
        # different things: a governorate with genuinely zero initial stock, and
        # one whose first bulletin happened to report no change because reporting
        # is lumpy. Both give y=0, but only the first is a structural zero. Using
        # the first strictly positive interval removes the spurious ones; a
        # governorate with no positive interval at all is a true zero.
        pos = np.flatnonzero(keep & (y > 0))
        anyk = np.flatnonzero(keep)
        if len(pos):
            k0 = pos[0]
            y_first[i] = float(y[k0])
            first_hold[i] = bool(k0 != anyk[0])
        elif len(anyk):
            k0 = anyk[0]
            y_first[i] = 0.0
            true_zero[i] = True
        else:
            continue
        cum_first[i] = 0.0 if not np.isfinite(cum[k0]) else cum[k0] / N
        t_first[i] = el[k0]

    if n_dupe:
        prov["duplicate_date_rows_merged"] = n_dupe
        if verbose:
            print(f"  merged {n_dupe} duplicate-date row(s)")
    if n_monotone:
        prov["nonmonotone_cum_rows_repaired"] = n_monotone
        if verbose:
            print(f"  repaired {n_monotone} non-monotone cumulative value(s) by "
                  f"running maximum (these would otherwise give negative increments)")
    if first_hold.any():
        prov["ic_anchor_skipped_leading_zero"] = [names[i] for i in
                                                  np.flatnonzero(first_hold)]
        if verbose:
            print("  IC anchored on the first POSITIVE interval for %d governorate(s) "
                  "whose first bulletin reported no change: %s"
                  % (int(first_hold.sum()),
                     ", ".join(prov["ic_anchor_skipped_leading_zero"])))
    if true_zero.any():
        prov["structural_zero_seed"] = [names[i] for i in np.flatnonzero(true_zero)]
        if verbose:
            print("  structural zero seed (no positive interval anywhere): %s"
                  % ", ".join(prov["structural_zero_seed"]))
    if not have_cum:
        prov["cum_first_unavailable"] = True
        warnings.warn("no cumulative column: the removed fraction at t=0 cannot be "
                      "reconstructed and is taken as 0 at every governorate.")

    rows = np.array(rows, float)
    gov = rows[:, 0].astype(int); t0 = rows[:, 1]; t1 = rows[:, 2]
    y_raw = np.maximum(rows[:, 3], 0.0); dC_raw = rows[:, 4]

    cut = float(np.quantile(y_raw, WINSOR_Q))
    if winsorize:
        y = np.minimum(y_raw, cut)
        n_w = int((y_raw > cut).sum())
        dC = np.minimum(dC_raw, cut * pop[gov] * (t1 - t0))
    else:
        y = y_raw.copy(); dC = dC_raw.copy(); n_w = 0

    w_count = 1.0 / np.sqrt(np.maximum(dC, 1.0)); w_count /= w_count.mean()
    w_rate = pop[gov] * (t1 - t0) / np.sqrt(np.maximum(dC, 1.0)); w_rate /= w_rate.mean()

    p = Panel(gov, t0, t1, y, y_raw, dC, dC_raw, w_count.copy(), w_count, w_rate,
              names, xg, pop, y_first, cum_first, t_first, cut, n_w,
              true_zero=true_zero, first_hold=first_hold,
              provenance=prov, synthetic=synthetic)
    return p


# --------------------------------------------------------------------------
# Observation design
# --------------------------------------------------------------------------
def build_interval_operator(grid: seip.Grid, panel: Panel):
    """Sparse (n_obs, (nt+1)*ng) matrix giving the interval AVERAGE of H over each
    reported interval -- what y_{g,k} = dC/(N dt) actually is."""
    from scipy.sparse import csr_matrix
    nt1 = grid.nt + 1; ng = panel.ng
    rows, cols, vals = [], [], []
    for r in range(panel.n_obs):
        k0 = int(np.floor(panel.t0[r] / grid.dt + 1e-9))
        k1 = int(np.ceil(panel.t1[r] / grid.dt - 1e-9))
        k0 = max(0, min(k0, grid.nt)); k1 = max(k0 + 1, min(k1, grid.nt))
        n = k1 - k0 + 1
        wt = np.full(n, grid.dt); wt[0] *= 0.5; wt[-1] *= 0.5
        wt /= wt.sum()
        g = panel.gov[r]
        for j, k in enumerate(range(k0, k1 + 1)):
            rows.append(r); cols.append(k * ng + g); vals.append(wt[j])
    return csr_matrix((vals, (rows, cols)), shape=(panel.n_obs, nt1 * ng))


def build_space_sampler(grid: seip.Grid, panel: Panel):
    """(ng, nx) linear interpolation from the x grid to governorate coordinates."""
    if grid.nx == 1:
        # Single location: without this branch the two interpolation writes land
        # on the same element and the second zeroes the first, giving an all-zero
        # sampler -- a silent failure in which a fit returns its seed with an
        # exactly zero gradient.
        return np.ones((panel.ng, 1))
    W = np.zeros((panel.ng, grid.nx))
    for i, xv in enumerate(panel.xg):
        xv = min(max(xv, grid.x[0]), grid.x[-1])
        j = min(int(np.floor(xv / grid.dx)), grid.nx - 2)
        frac = (xv - grid.x[j]) / grid.dx
        W[i, j] = 1.0 - frac
        W[i, j + 1] = frac
    return W


# --------------------------------------------------------------------------
# Synthetic twin
# --------------------------------------------------------------------------
# Governorate name, great-circle distance from Sana'a (km), population.
SYNTH_GOVS = [
    ("Amanat Al Asimah", 0.0, 2_950_000), ("Sana'a", 24.0, 1_240_000),
    ("Amran", 46.0, 1_090_000),           ("Dhamar", 72.0, 1_900_000),
    ("Al Mahwit", 88.0, 710_000),         ("Raymah", 106.0, 560_000),
    ("Hajjah", 121.0, 2_120_000),         ("Ibb", 137.0, 2_880_000),
    ("Al Hudaydah", 151.0, 3_190_000),    ("Al Jawf", 162.0, 570_000),
    ("Marib", 168.0, 330_000),            ("Al Bayda", 176.0, 740_000),
    ("Sa'ada", 199.0, 980_000),           ("Taizz", 204.0, 3_120_000),
    ("Al Dhale'e", 217.0, 660_000),       ("Lahj", 281.0, 980_000),
    ("Aden", 302.0, 940_000),             ("Abyan", 351.0, 570_000),
    ("Shabwah", 423.0, 650_000),          ("Hadramaut", 601.0, 1_420_000),
    ("Al Maharah", 795.396633, 150_000),
]
# Gaps between consecutive NATIONAL bulletin dates, with the record's weights.
SYNTH_GAPS = np.array([1, 2, 3, 4, 5, 7, 11, 14, 21])
SYNTH_GAP_P = np.array([0.5200, 0.3400, 0.0504, 0.0308, 0.0196, 0.0168, 0.0098, 0.0070, 0.0056])


def synthetic_truth(t, a10=0.030, a20=1.8e-5, tau=40.0, floor=0.49):
    """Declining transmission: R0 about 1.84 at t=0 falling to about 0.89, giving
    an epidemic with the record's shape -- roughly a 14-fold rise over the first
    six weeks, a peak, and a long non-zero tail."""
    shape = floor + (1.0 - floor) * np.exp(-np.asarray(t, float) / tau)
    return a10 * shape, a20 * shape


def make_synthetic_panel(grid: seip.Grid, truth=synthetic_truth, diff=None,
                         seed=11, t_max=seip.T_MAX, scheme="strang",
                         seed_scale=8.0e-5, decay_km=120.0, p_hold=0.18,
                         batch_dates=3, verbose=True, **ic_opts):
    """Generate a panel from a forward solve with a KNOWN transmission pair,
    sampled on the record's own sampling design.

    The cadence is SHARED across governorates, not drawn independently per
    governorate. That is what the record shows: 2,806 rows across only 136
    distinct bulletin days means essentially every governorate reports on
    essentially every national bulletin date. An independent-cadence twin
    produces ~272 distinct days and about a quarter of the record's exact-zero
    fraction, which matters because under the specified weight an exact zero
    carries the MAXIMUM possible weight -- so the zero rows dominate the
    objective, and a twin that under-represents them does not exercise the part
    of the objective being relied on.

    Reporting is also LUMPY, which is where most of the record's zeros come
    from. With probability `p_hold` a governorate reports no change at a
    bulletin and the cases accumulate into a backlog released at its next
    reporting bulletin. On `batch_dates` national dates every governorate
    releases simultaneously, reproducing the synchronized single-day spikes
    the record documents -- so the twin exercises the winsorization rule
    rather than assuming it.
    """
    rng = np.random.default_rng(seed)
    diff = diff or seip.Diffusion()
    names = [q[0] for q in SYNTH_GOVS]
    xg = np.array([q[1] for q in SYNTH_GOVS], float)
    pop = np.array([q[2] for q in SYNTH_GOVS], float)

    seed_rate = seed_scale * np.exp(-xg / decay_km)
    seed_rate[-1] = 0.0          # Al Maharah: zero initial stock, as in the record
    seed_rate[-2] = 0.0          # Hadramaut: likewise
    cum0 = 3.0e-4 * np.exp(-xg / decay_km)

    ic = seip.build_initial_condition(grid, xg, seed_rate, cum0, **ic_opts)
    b10, b20 = truth(grid.t)
    H, Sbar, _ = seip.solve_forward(grid, ic, b10, b20, diff, scheme=scheme)

    Wx = build_space_sampler(grid, type("P", (), {"ng": len(names), "xg": xg})())
    Hg = H @ Wx.T                                   # (nt+1, ng)

    # one national bulletin calendar, shared by every governorate
    dates = [0.0]
    while dates[-1] < t_max - 1:
        dates.append(min(dates[-1] + float(rng.choice(SYNTH_GAPS, p=SYNTH_GAP_P)), t_max))
    dates = np.array(sorted(set(dates)))

    # national dates on which every governorate releases its backlog at once
    forced = set(rng.choice(np.arange(1, len(dates)),
                            size=min(batch_dates, len(dates) - 1),
                            replace=False).tolist())
    recs = []
    for i, name in enumerate(names):
        cum = cum0[i] * pop[i]
        backlog = 0.0
        recs.append((name, dates[0], cum))
        for k in range(len(dates) - 1):
            t0, t1 = dates[k], dates[k + 1]
            k0 = int(round(t0 / grid.dt)); k1 = int(round(t1 / grid.dt))
            lam = max(Hg[k0:k1 + 1, i].mean() * pop[i] * (t1 - t0), 0.0)
            backlog += float(rng.poisson(lam)) if lam < 1e8 else lam
            if (k + 1) in forced or k == len(dates) - 2 or rng.random() >= p_hold:
                cum += backlog
                backlog = 0.0
            recs.append((name, t1, cum))

    df = pd.DataFrame(recs, columns=["governorate", "elapsed", "cumulative"])
    panel = _assemble(df, dict(zip(names, xg)), dict(zip(names, pop)),
                      winsorize=True, synthetic=True,
                      prov=dict(window_start="synthetic", shared_cadence=True))
    truth_d = {"beta10_t": b10, "beta20_t": b20, "diff": diff, "Sbar": Sbar,
               "seed_rate": seed_rate, "cum0": cum0, "scheme": scheme,
               "ic_opts": dict(ic_opts), "p_hold": p_hold,
               "batch_dates": sorted(forced), "decay_km": decay_km}
    if verbose:
        print("synthetic:", panel.describe())
        print("  record for comparison: 2806 rows, 21 governorates, 136 days, "
              "622 exact zeros (22.2%)")
    return panel, truth_d


# --------------------------------------------------------------------------
# Single-city synthetic twin
# --------------------------------------------------------------------------
# Three governorates of the second wave, by whole-epidemic suspected-case count
# (Camacho et al. 2018, Table 1) and by second-wave relative risk.
MAJOR_CITIES = {
    #  name                 x_km        population   cases     why
    "Al Hudaydah":      dict(x=151.0, pop=3_190_000, cases=163_533,
                             note="highest case count in the epidemic"),
    "Amanat Al Asimah": dict(x=0.0,   pop=2_950_000, cases=105_693,
                             note="the capital; singled out early in the second wave"),
    "Hajjah":           dict(x=121.0, pop=2_120_000, cases=123_658,
                             note="2nd highest count; among the top second-wave relative risks"),
    "Taizz":            dict(x=204.0, pop=3_120_000, cases=77_506,
                             note="among the top second-wave relative risks"),
    "Amran":            dict(x=46.0,  pop=1_090_000, cases=104_710,
                             note="highest attack rate after Al Mahwit"),
}


def city_truth(t, a10, a20, tau, floor):
    shape = floor + (1.0 - floor) * np.exp(-np.asarray(t, float) / tau)
    return a10 * shape, a20 * shape


def prefactors_from(R0_0, share):
    """Solve for (beta_10, beta_20) from a reproduction number at t=0 and an
    environmental share. Specifying the truth this way rather than by the two
    prefactors keeps the epidemic's SIZE and its ROUTE SPLIT independent -- the
    two things the comparison has to separate -- and the prefactors themselves
    are not comparable to each other anyway, since their kernels have different
    norms."""
    kap, J1, J2 = seip.ngm_integrals()
    tot = R0_0 / (kap * J1)
    return tot * (1.0 - share), tot * share / (seip.ALPHA0 * J2)


# Three trajectories differing in SIZE, SHAPE and ROUTE SPLIT, so a method that
# does well on one is not simply reproducing a shape common to all three. Each
# is tuned to the record's own profile: a 4-9% attack rate (Camacho et al. report
# 4.75-8.45% for the worst-hit governorates), a peak near day 60-70, and a
# roughly 8-19x rise over the first six weeks.
CITY_TRUTH = {
    "Al Hudaydah":      dict(R0_0=1.63, share=0.63, tau=55.0, floor=0.50),
    "Amanat Al Asimah": dict(R0_0=1.58, share=0.36, tau=48.0, floor=0.52),
    "Hajjah":           dict(R0_0=1.70, share=0.67, tau=55.0, floor=0.48),
    "Taizz":            dict(R0_0=1.55, share=0.50, tau=50.0, floor=0.51),
    "Amran":            dict(R0_0=1.66, share=0.45, tau=52.0, floor=0.50),
}


def city_beta(t, city):
    p = CITY_TRUTH[city]
    a10, a20 = prefactors_from(p["R0_0"], p["share"])
    return city_truth(t, a10, a20, p["tau"], p["floor"])


def make_city_panel(city, grid, seed=7, seed_rate=3.0e-5, cum0=3.0e-4,
                    p_hold=0.0, batch_dates=0, t_max=seip.T_MAX, verbose=True,
                    **ic_opts):
    """Generate one governorate's record from the SINGLE-LOCATION model (D = 0).

    The national twin is generated with diffusion on, so a city model fitted at
    D = 0 cannot reach the generating transmission pair -- the reduction itself
    is misspecified, and any recovery error then mixes architecture error with
    reduction error. Generating each city from the same reduction it will be
    fitted with makes the truth exactly attainable, so a comparison between two
    architectures measures the architectures.

    Sampling follows the record's bulletin calendar and the governorate's real
    population, with Poisson counts.

    p_hold DEFAULTS TO ZERO here, unlike the national twin. Batch reporting --
    a governorate reporting no change for several bulletins and then releasing
    the backlog -- is what produces most of the record's exact zeros, and at the
    national scale it must be reproduced. At a SINGLE governorate it destroys the
    likelihood: measured on this city at p_hold = 0.18, the generating truth
    itself scores a mean Poisson deviance of 1575 per observation, while a
    deterministic fit reaches 9.8 by distorting beta to chase the batch releases,
    returning a reproduction number peaking near 9.9. Any comparison run on such
    a panel measures which method chases reporting artifacts hardest, not which
    recovers transmission. With p_hold = 0 the truth scores 0.91 -- the value a
    well-specified Poisson model should score -- and the problem is well posed.
    Set p_hold > 0 deliberately, as a reporting-artifact sensitivity, not as the
    baseline.
    """
    if city not in MAJOR_CITIES:
        raise KeyError(f"{city!r} not in MAJOR_CITIES: {list(MAJOR_CITIES)}")
    meta = MAJOR_CITIES[city]
    N = float(meta["pop"])
    rng = np.random.default_rng(seed)

    g1 = seip.Grid(dt=grid.dt, nx=1)
    ic = seip.build_initial_condition(g1, np.array([0.0]), np.array([seed_rate]),
                                      np.array([cum0]), **ic_opts)
    b10, b20 = city_beta(g1.t, city)
    H, Sbar, _ = seip.solve_forward(g1, ic, b10, b20, seip.Diffusion(0, 0, 0, 0))
    Hc = H[:, 0]

    dates = [0.0]
    while dates[-1] < t_max - 1:
        dates.append(min(dates[-1] + float(rng.choice(SYNTH_GAPS, p=SYNTH_GAP_P)), t_max))
    dates = np.array(sorted(set(dates)))
    forced = set(rng.choice(np.arange(1, len(dates)),
                            size=min(batch_dates, len(dates) - 1), replace=False).tolist())

    recs = []
    cum = cum0 * N
    backlog = 0.0
    recs.append((city, dates[0], cum))
    for k in range(len(dates) - 1):
        t0, t1 = dates[k], dates[k + 1]
        k0, k1 = int(round(t0 / g1.dt)), int(round(t1 / g1.dt))
        lam = max(Hc[k0:k1 + 1].mean() * N * (t1 - t0), 0.0)
        backlog += float(rng.poisson(lam)) if lam < 1e8 else lam
        if (k + 1) in forced or k == len(dates) - 2 or rng.random() >= p_hold:
            cum += backlog; backlog = 0.0
        recs.append((city, t1, cum))

    df = pd.DataFrame(recs, columns=["governorate", "elapsed", "cumulative"])
    panel = _assemble(df, {city: meta["x"]}, {city: N}, winsorize=True,
                      synthetic=True, prov=dict(window_start="synthetic city",
                                                city=city, reduction="D=0"))
    truth = dict(beta10_t=b10, beta20_t=b20, Sbar=Sbar, H=Hc, grid=g1,
                 seed_rate=seed_rate, cum0=cum0, city=city,
                 params=CITY_TRUTH[city], ic_opts=dict(ic_opts))
    if verbose:
        print(f"{city}: {panel.describe()}")
    return panel, truth

In [ ]:
%%writefile fit.py
"""
fit.py -- Transmission-only estimation.

Only beta_10(t) and beta_20(t) are released. Everything else in seip.py is held;
D1..D4 are supplied and swept, never estimated here.

    g_j(t)  = c0 [+ cL t/T] + sum_m [cs_m sin(2 pi m t/T) + cc_m cos(2 pi m t/T)]
    beta_j0 = softplus(g_j) + eps

M is bounded above by the sampling: a harmonic of index m has period T/m and
cannot be resolved from samples more than half its period apart, so
M <= T / (2 * max interval).

DECLARED DEVIATIONS from the written methodology, all deliberate:
  * the smoothness penalty is applied to the RELATIVE derivative by default
    (smooth_scale="relative"). The absolute form (1/T)int[b10'^2 + b20'^2] is
    scale dependent, and with b10 ~ 1e-2 against b20 ~ 1e-5 it regularizes b10
    about a million times harder in relative terms. smooth_scale="absolute"
    gives the specified form.
  * the ridge spares the constant and trend coefficients and penalizes only the
    harmonics, so it shrinks structure rather than level.
  * age quadrature is composite trapezoid on the solver grid rather than
    Gauss-Legendre split at h's breakpoints. Benign at dt <= 0.25 -- both
    breakpoints land on nodes and h is continuous at both -- and
    seip.ngm_integrals(grid, report=True) quantifies it.
"""

from __future__ import annotations

import os
import numpy as np
from dataclasses import dataclass, field
from scipy.optimize import least_squares

import seip
import data_io

EPS_BETA = 1e-12
FLOOR_TOL = 1e-9          # below this beta is on the positivity floor


def softplus(z):
    return np.logaddexp(0.0, z)


def inv_softplus(v):
    v = np.asarray(v, float)
    return np.where(v > 20.0, v, np.log(np.expm1(np.minimum(v, 20.0))))


def max_fourier_order(panel: data_io.Panel, T=seip.T_MAX):
    """Nyquist ceiling on M from the worst-served governorate and from the pooled
    bulletin calendar."""
    gap_local = float((panel.t1 - panel.t0).max())
    days = np.unique(np.round(np.concatenate([panel.t0, panel.t1]), 6))
    gap_pooled = float(np.diff(days).max()) if len(days) > 1 else gap_local
    return T / (2.0 * gap_local), T / (2.0 * gap_pooled)


@dataclass
class BetaModel:
    """M=0 with trend=True is a constant PLUS a linear trend. trend=False, M=0 is
    the strict constant."""
    M: int = 0
    T: float = seip.T_MAX
    trend: bool = True

    @property
    def nper(self):
        return 1 + int(self.trend) + 2 * self.M

    @property
    def nparam(self):
        return 2 * self.nper

    def names(self):
        base = ["const"] + (["trend"] if self.trend else [])
        for m in range(1, self.M + 1):
            base += ["sin%d" % m, "cos%d" % m]
        return ["b10:" + n for n in base] + ["b20:" + n for n in base]

    def design(self, t):
        t = np.asarray(t, float)
        cols = [np.ones_like(t)]
        if self.trend:
            cols.append(t / self.T)
        for m in range(1, self.M + 1):
            cols.append(np.sin(2 * np.pi * m * t / self.T))
            cols.append(np.cos(2 * np.pi * m * t / self.T))
        return np.column_stack(cols)

    def eval(self, theta, t, X=None):
        X = self.design(t) if X is None else X
        return (softplus(X @ theta[:self.nper]) + EPS_BETA,
                softplus(X @ theta[self.nper:]) + EPS_BETA)

    def start(self, b10, b20):
        th = np.zeros(self.nparam)
        th[0] = float(inv_softplus(b10))
        th[self.nper] = float(inv_softplus(b20))
        return th

    def harmonic_mask(self):
        k = 1 + int(self.trend)
        m = np.zeros(self.nparam, bool)
        m[k:self.nper] = True
        m[self.nper + k:] = True
        return m


@dataclass
class FitResult:
    label: str
    diff: seip.Diffusion
    theta: np.ndarray
    beta10_t: np.ndarray
    beta20_t: np.ndarray
    R0_t: np.ndarray
    Sbar: np.ndarray
    H: np.ndarray
    Hg: np.ndarray
    yhat: np.ndarray
    resid_w: np.ndarray
    wrmse: float
    rmse: float
    rel_mae: float
    nfev: int
    success: bool
    status: int = 0
    meta: dict = field(default_factory=dict)

    @property
    def converged(self):
        """status > 0 means a real termination criterion fired. status == 0 is
        'max_nfev reached', which is not convergence and must not be tabulated or
        plotted as if it were."""
        return bool(self.success) and self.status > 0

    @property
    def saturation(self):
        """Fraction of the window on which each prefactor sits on the positivity
        floor. Anything above a few per cent means the reported curve is the
        constraint, not an estimate, and the Jacobian column there is ~0 -- so
        success=True says nothing about that parameter."""
        return self.meta["sat_b10"], self.meta["sat_b20"]


class Problem:
    """Everything fixed across a fit: grid, panel, IC, operators, normalization."""

    def __init__(self, grid: seip.Grid, panel: data_io.Panel,
                 truncation_correction=False, age_profile="flat",
                 p_from_model=False, row_mask=None, scheme="strang",
                 observable=seip.OBS_FORCE_OF_INFECTION, yscale=None,
                 objective="wls"):
        """row_mask selects which rows enter the DATA TERM. The PDE still runs
        over the whole domain, so a per-governorate fit under live diffusion is a
        different object from one in isolation.

        yscale may be supplied so several Problems sharing a panel normalize
        identically. Without it a masked Problem puts its objective -- and any
        gradient taken from it -- in its own units, which makes cross-location
        comparison meaningless.
        """
        self.grid = grid
        self.panel = panel
        self.scheme = scheme
        self.observable = observable
        self.row_mask = (np.ones(panel.n_obs, bool) if row_mask is None
                         else np.asarray(row_mask, bool))
        self.ic = seip.build_initial_condition(
            grid, panel.xg, panel.y_first, panel.cum_first,
            truncation_correction=truncation_correction,
            age_profile=age_profile, p_from_model=p_from_model)
        self.Wx = data_io.build_space_sampler(grid, panel)
        self.A = data_io.build_interval_operator(grid, panel)
        # Exact next-generation integrals, not this grid's trapezoid values: they
        # define R0 and the published constants are the exact ones.
        self.kappa, self.J1, self.J2 = seip.ngm_integrals()
        m = self.row_mask
        self.yscale = (float(np.sqrt(np.mean((panel.w[m] * panel.y[m]) ** 2)))
                       if yscale is None else float(yscale))
        # With yscale taken from this same mask, sse_null is identically m.sum():
        # a row count, not a measured quantity. sse_floor is the informative one.
        self.sse_null = float(np.sum((panel.w[m] * panel.y[m] / self.yscale) ** 2))
        var_y = panel.dC[m] / np.maximum(
            (panel.pop[panel.gov[m]] * (panel.t1[m] - panel.t0[m])) ** 2, 1e-300)
        self.sse_floor = float(np.sum(panel.w[m] ** 2 * var_y) / self.yscale ** 2)
        self.objective = objective
        # Scale the deviance so the two objectives sit on comparable footing:
        # sqrt(n) makes a well-specified Poisson fit land near 1 per residual.
        self.dev_scale = float(np.sqrt(max(int(m.sum()), 1)))
        self.ic_opts = dict(truncation_correction=truncation_correction,
                            age_profile=age_profile, p_from_model=p_from_model)

    def predict(self, beta10_t, beta20_t, diff: seip.Diffusion):
        H, Sbar, _ = seip.solve_forward(self.grid, self.ic, beta10_t, beta20_t,
                                        diff, scheme=self.scheme,
                                        observable=self.observable)
        Hg = H @ self.Wx.T
        return H, Hg, self.A @ Hg.ravel(), Sbar

    def data_residual(self, yhat):
        """objective='wls'      w (yhat - y) / yscale, the specified form.
        objective='poisson'  the Poisson DEVIANCE residual on COUNTS,

            d = sign(c - lam) sqrt( 2 [ c log(c/lam) - (c - lam) ] ),
            lam = yhat N dt,   with c log(c/lam) := 0 at c = 0 so d = sqrt(2 lam).

        The deviance is the right object here and weighted least squares is not.
        1/sqrt(max(c,1)) estimates the variance from the realization, so a row
        that fluctuated low is up-weighted and an exact zero maximally; on a panel
        with the record's 22% exact-zero fraction the zero rows carry ~82% of the
        objective, and what 82% of the objective asks for is 'predict zero'. The
        deviance has no such pathology: a zero row contributes 2*lam, which
        penalizes over-prediction without rewarding collapse.
        """
        m = self.row_mask
        if self.objective == "wls":
            return self.panel.w[m] * (yhat[m] - self.panel.y[m]) / self.yscale
        Ndt = self.panel.pop[self.panel.gov[m]] * (self.panel.t1[m] - self.panel.t0[m])
        lam = np.maximum(np.asarray(yhat)[m] * Ndt, 1e-10)
        c = self.panel.dC[m]
        term = np.where(c > 0, c * np.log(np.maximum(c, 1e-300) / lam), 0.0)
        d2 = 2.0 * (term - (c - lam))
        return np.sign(c - lam) * np.sqrt(np.maximum(d2, 0.0)) / self.dev_scale

    def residuals(self, theta, bm, diff, X, lam_smooth, lam_ridge,
                  smooth_scale="relative"):
        b10, b20 = bm.eval(theta, self.grid.t, X)
        _, _, yhat, _ = self.predict(b10, b20, diff)
        parts = [self.data_residual(yhat)]
        if lam_smooth > 0 and bm.nper > 1:
            dt = self.grid.dt
            d10, d20 = np.diff(b10) / dt, np.diff(b20) / dt
            if smooth_scale == "relative":
                d10 = d10 / max(b10.mean(), 1e-300)
                d20 = d20 / max(b20.mean(), 1e-300)
            s = np.sqrt(lam_smooth * dt / seip.T_MAX)
            parts += [s * d10, s * d20]
        if lam_ridge > 0 and bm.M > 0:
            parts.append(np.sqrt(lam_ridge) * theta[bm.harmonic_mask()])
        return np.concatenate(parts)

    def penalty_share(self, theta, bm, diff, X, lam_smooth, lam_ridge, smooth_scale):
        """Fraction of the objective contributed by the penalties. The
        methodology calls the smoothness term 'a prior with content'; if this is
        1e-5 the prior has no content at these settings and should be reported as
        inert rather than as doing work."""
        nd = int(self.row_mask.sum())
        r = self.residuals(theta, bm, diff, X, lam_smooth, lam_ridge, smooth_scale)
        tot = float(r @ r)
        return float(r[nd:] @ r[nd:]) / max(tot, 1e-300)


class _Residual:
    """Module level so it pickles for scipy's parallel Jacobian."""

    def __init__(self, prob, bm, diff, X, lam_smooth, lam_ridge, smooth_scale):
        self.prob, self.bm, self.diff, self.X = prob, bm, diff, X
        self.lam_smooth, self.lam_ridge = lam_smooth, lam_ridge
        self.smooth_scale = smooth_scale

    def __call__(self, theta):
        return self.prob.residuals(theta, self.bm, self.diff, self.X,
                                   self.lam_smooth, self.lam_ridge,
                                   self.smooth_scale)


# --------------------------------------------------------------------------
# Coarse search: seeds the optimizer AND is the route-split diagnostic
# --------------------------------------------------------------------------
def grid_search(prob: Problem, diff: seip.Diffusion,
                b10_range=(1e-5, 3e-1), b20_range=(1e-7, 1e-3), n=13, verbose=True):
    """Returns (b10s, b20s, sse, best). Warns on a boundary optimum: that means
    the box chose the seed, not the data."""
    b10s = np.geomspace(*b10_range, n)
    b20s = np.geomspace(*b20_range, n)
    sse = np.full((n, n), np.nan)
    nt1 = prob.grid.nt + 1
    m = prob.row_mask
    for i, a in enumerate(b10s):
        for j, b in enumerate(b20s):
            _, _, yh, _ = prob.predict(np.full(nt1, a), np.full(nt1, b), diff)
            r = prob.data_residual(yh)          # honours prob.objective
            sse[i, j] = float(r @ r)
    k = np.unravel_index(np.nanargmin(sse), sse.shape)
    edge = [nm for nm, idx in (("beta10", k[0]), ("beta20", k[1]))
            if idx in (0, n - 1)]
    if verbose:
        print("  grid search best: beta10=%.4g beta20=%.4g SSE=%.6g"
              % (b10s[k[0]], b20s[k[1]], sse[k]))
        if edge:
            print("    WARNING: optimum on the box edge in %s -- widen the range; "
                  "the seed is set by the box, not the data" % ", ".join(edge))
    return b10s, b20s, sse, (b10s[k[0]], b20s[k[1]])


# --------------------------------------------------------------------------
# Uncertainty from the Jacobian at the solution
# --------------------------------------------------------------------------
def parameter_uncertainty(sol, bm: BetaModel, n_data: int):
    """J^T J at the solution IS the Gauss-Newton Hessian the methodology asks
    for. scipy returns it for free; earlier versions discarded it.

    The standard errors assume independent, correctly scaled residuals, which the
    error surface shows is false, so they are a LOWER bound on the true spread.
    The eigenvalue structure -- in particular the near-null direction -- is the
    part that does not depend on that assumption.
    """
    J = np.asarray(sol.jac, float)
    H = J.T @ J
    ev, evec = np.linalg.eigh(H)
    order = np.argsort(ev)[::-1]
    ev, evec = ev[order], evec[:, order]
    dof = max(n_data - bm.nparam, 1)
    s2 = 2.0 * float(sol.cost) / dof
    with np.errstate(divide="ignore", invalid="ignore"):
        cov = s2 * np.linalg.pinv(H)
        se = np.sqrt(np.clip(np.diag(cov), 0.0, None))
    cond = float(ev[0] / ev[-1]) if ev[-1] > 0 else np.inf
    return dict(eigenvalues=ev, eigenvectors=evec, cond=cond, se=se, sigma2=s2,
                names=bm.names(), null_direction=evec[:, -1],
                null_eigenvalue=float(ev[-1]))


# --------------------------------------------------------------------------
# The fit
# --------------------------------------------------------------------------
def fit(prob: Problem, diff: seip.Diffusion, M=0, lam_smooth=1e-2, lam_ridge=1e-4,
        seed_beta=None, label=None, smooth_scale="relative", max_nfev=400,
        trend=True, n_jobs=1, irls=0, verbose=True):
    """irls > 0 runs that many extra passes with MODEL-based weights
    1/sqrt(max(yhat N dt, 1)) rebuilt from the previous fit.

    Why it matters: the specified weight 1/sqrt(observed count) uses the
    realization as its own variance estimate, so a row that fluctuated low is
    up-weighted and an exact zero maximally so. On a panel matching the record's
    22% exact-zero fraction the zero rows carry about 82% of the objective. That
    is the downward bias of Poisson-weighted least squares, and it pushes in the
    SAME direction as the initial-condition truncation mismatch.
    """
    bm = BetaModel(M=M, trend=trend)
    X = bm.design(prob.grid.t)
    if seed_beta is None:
        _, _, _, seed_beta = grid_search(prob, diff, verbose=verbose)
    theta0 = bm.start(*seed_beta)

    kw = {}
    if n_jobs and n_jobs != 1:
        kw["workers"] = (os.cpu_count() if n_jobs < 0 else n_jobs)

    passes = [prob.panel.weight_kind]
    sol = None
    for pi in range(1 + max(0, irls)):
        fun = _Residual(prob, bm, diff, X, lam_smooth, lam_ridge, smooth_scale)
        sol = least_squares(fun, theta0, method="trf", x_scale="jac",
                            diff_step=1e-4, max_nfev=max_nfev, verbose=0, **kw)
        theta0 = sol.x
        if pi < max(0, irls):
            b10, b20 = bm.eval(sol.x, prob.grid.t, X)
            _, _, yh, _ = prob.predict(b10, b20, diff)
            prob.panel.set_weights("model", yhat=np.maximum(yh, 0.0))
            passes.append("model")
            m = prob.row_mask
            prob.yscale = float(np.sqrt(np.mean(
                (prob.panel.w[m] * prob.panel.y[m]) ** 2)))

    b10, b20 = bm.eval(sol.x, prob.grid.t, X)
    H, Hg, yhat, Sbar = prob.predict(b10, b20, diff)
    R0 = seip.R0_of_t(b10, b20, Sbar, prob.kappa, prob.J1, prob.J2)
    share, ratio, share_t = seip.route_share(b10, b20, prob.J2, prob.grid)

    m = prob.row_mask
    w, y, yh = prob.panel.w[m], prob.panel.y[m], yhat[m]
    wrmse = float(np.sqrt(np.mean((w * (yh - y)) ** 2)))
    rmse = float(np.sqrt(np.mean((yh - y) ** 2)))
    denom = np.maximum(y, np.quantile(y[y > 0], 0.10) if (y > 0).any() else 1.0)
    rel_mae = float(np.mean(np.abs(yh - y) / denom))
    rd = prob.data_residual(yhat)
    sse = float(rd @ rd)
    # 'explained' is built from the WLS null and Poisson-noise-floor sums of
    # squares. Those are not commensurate with a deviance, so it is reported only
    # under the WLS objective and left undefined otherwise rather than printed as
    # a large negative number.
    if prob.objective == "wls":
        rn = prob.panel.w[m] * prob.panel.y[m] / prob.yscale
        sse_wls = float(np.sum((prob.panel.w[m] * (yh - y) / prob.yscale) ** 2))
        span = prob.sse_null - prob.sse_floor
        explained = ((prob.sse_null - sse_wls) / span
                     if span > 1e-9 * max(prob.sse_null, 1e-300) else float("nan"))
    else:
        explained = float("nan")

    res = FitResult(
        label=label or diff.label(), diff=diff, theta=sol.x,
        beta10_t=b10, beta20_t=b20, R0_t=R0, Sbar=Sbar, H=H, Hg=Hg, yhat=yhat,
        resid_w=w * (yh - y), wrmse=wrmse, rmse=rmse, rel_mae=rel_mae,
        nfev=int(sol.nfev), success=bool(sol.success), status=int(sol.status),
        meta=dict(M=M, trend=trend, explained=explained, sse=sse,
                  sse_null=prob.sse_null, sse_floor=prob.sse_floor,
                  env_share=share, route_ratio=ratio, env_share_t=share_t,
                  sat_b10=float(np.mean(b10 <= FLOOR_TOL)),
                  sat_b20=float(np.mean(b20 <= FLOOR_TOL)),
                  weight_kind=prob.panel.weight_kind, weight_passes=passes,
                  lam_smooth=lam_smooth, lam_ridge=lam_ridge, seed=seed_beta,
                  cost=float(sol.cost), scheme=prob.scheme,
                  observable=prob.observable, objective=prob.objective,
                  penalty_share=prob.penalty_share(sol.x, bm, diff, X, lam_smooth,
                                                   lam_ridge, smooth_scale),
                  uncertainty=parameter_uncertainty(sol, bm, int(m.sum())),
                  **prob.ic_opts))
    if verbose:
        flag = "" if res.converged else "   [NOT CONVERGED: max_nfev]"
        sat = res.meta["sat_b10"]
        satf = ("   [b10 on the floor %.0f%% of the window]" % (100 * sat)
                if sat > 0.02 else "")
        print("  [%s] M=%d%s wRMSE=%.4g relMAE=%.3f expl=%.3f envshare=%.3f "
              "R0 %.2f-%.2f cond=%.1e nfev=%d%s%s"
              % (res.label, M, "+trend" if trend else "", wrmse, rel_mae,
                 explained, share, R0.min(), R0.max(),
                 res.meta["uncertainty"]["cond"], sol.nfev, flag, satf))
    return res


def sweep_diffusion(prob: Problem, diff_list, M=0, seed_beta=None, verbose=True, **kw):
    """Refit the transmission pair under each diffusion setting. D is held and
    swept, never estimated, as the staged argument requires."""
    out = []
    for d in diff_list:
        if verbose:
            print("fitting %s ..." % d.label())
        out.append(fit(prob, d, M=M, seed_beta=seed_beta, label=d.label(),
                       verbose=verbose, **kw))
    return out


# --------------------------------------------------------------------------
# Held-out validation
# --------------------------------------------------------------------------
def blocked_time_folds(panel: data_io.Panel, k=5):
    """Contiguous blocks of bulletin time. A random row split would leak:
    neighbouring intervals at one governorate are nearly the same observation."""
    edges = np.quantile(panel.t1, np.linspace(0, 1, k + 1))
    edges[0] -= 1e-9; edges[-1] += 1e-9
    return [(panel.t1 > edges[i]) & (panel.t1 <= edges[i + 1]) for i in range(k)]


PROB_KEYS = ("truncation_correction", "age_profile", "p_from_model",
             "scheme", "observable", "objective")


def blocked_validation(grid, panel, diff, M=0, k=5, seed_beta=None, verbose=True,
                       **kw):
    """Fit on all but one contiguous time block; score on the held-out block."""
    rows = []
    for i, test in enumerate(blocked_time_folds(panel, k)):
        train = ~test
        prob = Problem(grid, panel, row_mask=train,
                       **{z: kw[z] for z in PROB_KEYS if z in kw})
        fk = {z: v for z, v in kw.items() if z not in PROB_KEYS}
        r = fit(prob, diff, M=M, seed_beta=seed_beta, label="fold%d" % i,
                verbose=False, **fk)
        w, y, yh = panel.w[test], panel.y[test], r.yhat[test]
        rows.append(dict(fold=i, n_train=int(train.sum()), n_test=int(test.sum()),
                         t_lo=float(panel.t1[test].min()),
                         t_hi=float(panel.t1[test].max()),
                         train_wrmse=r.wrmse,
                         test_wrmse=float(np.sqrt(np.mean((w * (yh - y)) ** 2))),
                         R0_mean=float(r.R0_t.mean()),
                         beta10_mean=float(r.beta10_t.mean()),
                         env_share=r.meta["env_share"], converged=r.converged))
        if verbose:
            q = rows[-1]
            print("  fold %d: test days %.0f-%.0f  train wRMSE %.3g  "
                  "test wRMSE %.3g  R0bar %.3f"
                  % (i, q["t_lo"], q["t_hi"], q["train_wrmse"], q["test_wrmse"],
                     q["R0_mean"]))
    return rows


def multistart(prob, diff, seeds, M=0, verbose=True, **kw):
    """Refit from several seeds. The objective is non-convex and the route-split
    direction is near flat, so the SPREAD across seeds is a reported quantity,
    not a diagnostic to pass."""
    out = []
    for i, sb in enumerate(seeds):
        r = fit(prob, diff, M=M, seed_beta=sb, label="seed%d" % i,
                verbose=False, **kw)
        out.append(r)
        if verbose:
            print("  seed %d from (%.3g, %.3g): b10bar=%.4g b20bar=%.4g "
                  "R0bar=%.4f wRMSE=%.4g%s"
                  % (i, sb[0], sb[1], r.beta10_t.mean(), r.beta20_t.mean(),
                     r.R0_t.mean(), r.wrmse,
                     "" if r.converged else "  [not converged]"))
    return out


# --------------------------------------------------------------------------
# Fields for plotting
# --------------------------------------------------------------------------
def observed_field(panel: data_io.Panel, grid: seip.Grid):
    """(nt+1, ng) observed per-capita incidence, NaN where no bulletin interval
    covers that time. Each interval is drawn across its own span -- exactly what
    a bulletin states. Nothing is interpolated between bulletins; gaps occur only
    before a governorate's first report."""
    Y = np.full((grid.nt + 1, panel.ng), np.nan)
    for r in range(panel.n_obs):
        k0 = int(np.floor(panel.t0[r] / grid.dt + 1e-9))
        k1 = int(np.ceil(panel.t1[r] / grid.dt - 1e-9))
        Y[k0:k1 + 1, panel.gov[r]] = panel.y[r]
    return Y


def model_interval_field(panel: data_io.Panel, grid: seip.Grid, res: FitResult):
    """The model on the SAME footing as the observations: each reported interval
    filled with the model's interval AVERAGE, which is the quantity actually
    minimized. Comparing the instantaneous operator against a 21-day interval
    average compares two different objects."""
    Yh = np.full((grid.nt + 1, panel.ng), np.nan)
    for r in range(panel.n_obs):
        k0 = int(np.floor(panel.t0[r] / grid.dt + 1e-9))
        k1 = int(np.ceil(panel.t1[r] / grid.dt - 1e-9))
        Yh[k0:k1 + 1, panel.gov[r]] = res.yhat[r]
    return Yh


def national_curve(panel: data_io.Panel):
    """National daily incidence from the ROWS, not from a filled field: for each
    bulletin interval, total cases divided by total population and interval
    length. Returns (t_mid, cases_per_day, rate). No gap filling anywhere."""
    order = np.argsort(0.5 * (panel.t0 + panel.t1))
    tm = 0.5 * (panel.t0 + panel.t1)[order]
    cases = panel.dC[order]
    dt_ = (panel.t1 - panel.t0)[order]
    uniq = np.unique(np.round(tm, 6))
    cpd = np.array([cases[np.round(tm, 6) == u].sum()
                    / max(dt_[np.round(tm, 6) == u].mean(), 1e-9) for u in uniq])
    rate = cpd / panel.pop.sum()
    return uniq, cpd, rate

In [ ]:
%%writefile pinn.py
"""
pinn.py -- Dual-network physics-informed neural networks for the age-structured
SEIP system, in the two variants this project needs to compare.

ARCHITECTURES (both are DUAL networks, per this project's own Sec. 10 design:
one network for the states, a second for the parameters, coupled only through a
SOFT residual penalty -- the coupling whose failure mode the per-region memo
diagnosed).

    DPINN    FieldNet: plain tanh MLP on the normalized coordinates -> softplus
                       outputs (S, E, I, P).
             BetaNet:  plain tanh MLP on normalized t -> softplus (b10, b20).

    FFPINN   FieldNet: the SAME MLP, but each normalized coordinate is first
                       lifted by a deterministic Fourier feature map
                           Phi_K(u) = [u, sin(pi u), cos(pi u), ..., sin(K pi u), cos(K pi u)]
             BetaNet:  a truncated Fourier series with a linear trend, passed
                       through softplus.

The two changes point in OPPOSITE directions and the paper must say so. The
embedding on FieldNet RAISES the frequency content the state representation can
reach (a coordinate MLP is biased toward low frequencies). The series on BetaNet
LOWERS it -- it is a hard low-pass restriction to M harmonics, with M bounded by
the sampling: a harmonic of index m has period T/m and cannot be resolved from
samples more than half its period apart, so M <= T/(2*max gap) = 6.5 here.

OBJECTIVE. The data term is the Poisson deviance on counts, not weighted least
squares. On a panel matching the record's 22% exact-zero fraction the specified
weight 1/sqrt(count) puts ~82% of the objective on zero rows, and a fit then
collapses toward predicting zero; see weighting_test.py. Weighted MSE is kept as
`objective="wls"` for the declared comparison.
"""
from __future__ import annotations

import math
import numpy as np
import torch
import torch.nn as nn

import seip

torch.set_default_dtype(torch.float64)
DEV = torch.device("cpu")


# --------------------------------------------------------------------------
# Quadrature
# --------------------------------------------------------------------------
def gauss_legendre(a, b, n):
    x, w = np.polynomial.legendre.leggauss(n)
    return 0.5 * (b - a) * x + 0.5 * (b + a), 0.5 * (b - a) * w


def age_nodes(n_a=12):
    """Gauss-Legendre on [0, A_MAX] for the infection-age integrals."""
    return gauss_legendre(0.0, seip.A_MAX, n_a)


def path_nodes(n1=4, n2=4, n3=10):
    """Gauss-Legendre on [0,1], [1,2], [2,30] so h(b)'s two breakpoints are not
    straddled by a single rule -- the split the methodology specifies. h is
    constant on the last panel and P decays smoothly there, so a short rule is
    accurate; the node counts are checked against the exact J2 below."""
    xs, ws = [], []
    for (lo, hi, n) in ((0.0, 1.0, n1), (1.0, 2.0, n2), (2.0, seip.B_MAX, n3)):
        x, w = gauss_legendre(lo, hi, n)
        xs.append(x); ws.append(w)
    return np.concatenate(xs), np.concatenate(ws)


# --------------------------------------------------------------------------
# Networks
# --------------------------------------------------------------------------
class FourierEmbed(nn.Module):
    """Deterministic Fourier feature map applied PER COORDINATE.

    Deterministic rather than random: with one input dimension per physical
    coordinate and a modest K, the deterministic ladder spans the same band
    without a seed-dependent draw, so a DPINN/FF-PINN comparison is not
    confounded by which random matrix was sampled.
    """

    def __init__(self, orders):
        super().__init__()
        self.orders = list(orders)

    @property
    def out_dim(self):
        return sum(1 + 2 * k for k in self.orders)

    def forward(self, z):
        out = []
        for j, K in enumerate(self.orders):
            u = z[:, j:j + 1]
            out.append(u)
            for m in range(1, K + 1):
                out.append(torch.sin(m * math.pi * u))
                out.append(torch.cos(m * math.pi * u))
        return torch.cat(out, dim=1)


class MLP(nn.Module):
    def __init__(self, d_in, d_out, width=64, depth=4):
        super().__init__()
        layers, d = [], d_in
        for _ in range(depth):
            layers += [nn.Linear(d, width), nn.Tanh()]
            d = width
        layers += [nn.Linear(d, d_out)]
        self.net = nn.Sequential(*layers)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight, gain=1.0)
                nn.init.zeros_(m.bias)

    def forward(self, z):
        return self.net(z)


class RawNet(nn.Module):
    """Unconstrained scalar output; positivity is imposed by the HardIC wrapper."""

    def __init__(self, d_in, width=64, depth=4, fourier_orders=None):
        super().__init__()
        self.embed = FourierEmbed(fourier_orders) if fourier_orders else None
        d = self.embed.out_dim if self.embed else d_in
        self.mlp = MLP(d, 1, width, depth)

    def forward(self, z):
        h = self.embed(z) if self.embed else z
        return self.mlp(h).squeeze(-1)


class FieldNet(nn.Module):
    """Normalized coordinates -> one non-negative field.

    Softplus output: smooth everywhere, so the derivatives the physics loss
    needs are not corrupted by a kink, and non-negative by architecture rather
    than by clamping, which would cut the gradient path autograd depends on.
    """

    def __init__(self, d_in, width=64, depth=4, fourier_orders=None, scale=1.0):
        super().__init__()
        self.embed = FourierEmbed(fourier_orders) if fourier_orders else None
        d = self.embed.out_dim if self.embed else d_in
        self.mlp = MLP(d, 1, width, depth)
        self.log_scale = nn.Parameter(torch.tensor(math.log(scale)))

    def forward(self, z):
        h = self.embed(z) if self.embed else z
        return torch.exp(self.log_scale) * nn.functional.softplus(self.mlp(h)).squeeze(-1)


# --------------------------------------------------------------------------
# Architecture registry: the 2x2 design
#
#                       beta = free MLP      beta = order-M Fourier series
#   states plain MLP    "dpinn"              "dpinn_fs"
#   states Fourier      "fdpinn"             "ff"
#
# "dpinn" and "ff" are the two arms this project reported first; because they
# differ in BOTH factors at once, neither effect was attributable from them.
# The two added cells complete the factorial.
# --------------------------------------------------------------------------
ARCH = {
    "dpinn":    (False, "mlp"),        # DPINN
    "dpinn_fs": (False, "fourier"),    # DPINN-FS
    "fdpinn":   (True,  "mlp"),        # F-DPINN
    "ff":       (True,  "fourier"),    # FF-PINN
}
ARCH_NAME = {
    "dpinn":    "DPINN (plain states, MLP beta)",
    "dpinn_fs": "DPINN-FS (plain states, Fourier beta)",
    "fdpinn":   "F-DPINN (Fourier states, MLP beta)",
    "ff":       "FF-PINN (Fourier states, Fourier beta)",
}


class BetaNetMLP(nn.Module):
    """DPINN parameter network: normalized t -> (b10, b20), softplus, with NO
    hand-picked multiplicative scale on the output. A learned log-scale is used
    instead so the magnitude comes from the loss rather than from a constant
    chosen before the first gradient step."""

    def __init__(self, width=48, depth=3, scale10=1e-2, scale20=1e-5):
        super().__init__()
        self.mlp = MLP(1, 2, width, depth)
        self.ls = nn.Parameter(torch.tensor([math.log(scale10), math.log(scale20)]))

    def forward(self, tau):
        out = nn.functional.softplus(self.mlp(tau))
        return (torch.exp(self.ls[0]) * out[:, 0] + 1e-14,
                torch.exp(self.ls[1]) * out[:, 1] + 1e-14)


class BetaNetFourier(nn.Module):
    """FF-PINN parameter model: truncated Fourier series with a linear trend.

    The trend is present because a pure Fourier basis is periodic at the domain
    edges and would otherwise force the first and last fitted values to agree.
    M is Nyquist-bounded by the worst-served governorate's reporting gap.
    """

    def __init__(self, M=4, scale10=1e-2, scale20=1e-5):
        super().__init__()
        self.M = M
        self.c = nn.Parameter(torch.zeros(2, 2 + 2 * M))
        with torch.no_grad():
            self.c[0, 0] = math.log(math.expm1(1.0))
            self.c[1, 0] = math.log(math.expm1(1.0))
        self.ls = nn.Parameter(torch.tensor([math.log(scale10), math.log(scale20)]))

    def basis(self, tau):
        t = 0.5 * (tau + 1.0)                       # tau in [-1,1] -> t/T in [0,1]
        cols = [torch.ones_like(t), t]
        for m in range(1, self.M + 1):
            cols += [torch.sin(2 * math.pi * m * t), torch.cos(2 * math.pi * m * t)]
        return torch.cat(cols, dim=1)

    def forward(self, tau):
        X = self.basis(tau)
        g = X @ self.c.T
        out = nn.functional.softplus(g)
        return (torch.exp(self.ls[0]) * out[:, 0] + 1e-14,
                torch.exp(self.ls[1]) * out[:, 1] + 1e-14)


class HardIC(nn.Module):
    """Wraps a raw network so the initial condition holds EXACTLY, by
    construction, instead of being enforced by a penalty.

    Why this matters here and is not a numerical nicety. The problem statement's
    Sec. 5 amplitude scaling

        (E, I, P, b10, b20) -> (cE, cI, cP, b10/c, b20/c)

    leaves the observation operator and every transport equation invariant and
    is broken only by the renewal condition for E. In a soft-penalty PINN that
    relation is one more residual term, so c is nearly free and the LEVEL of the
    transmission estimate is set by the loss weights rather than by the data.
    Pinning the initial condition exactly removes c: measured on the city twin, a
    soft IC returns a reproduction number inflated by a factor of 2 to 3 with the
    states correspondingly too small, while this parameterization does not.

        ramp(t) = 1 - exp(-t/lam)          ramp(0) = 0

        zero IC      u = ramp * scale * softplus(net)
        positive IC  u = u0 * exp(ramp * clamp(net))
        decreasing   u = u0 * exp(-ramp * softplus(net))      (used for S)

    Each is non-negative by construction and exact at t = 0.
    """

    def __init__(self, net, mode, u0=None, scale=1.0, lam=2.0, clamp=12.0):
        super().__init__()
        self.net = net
        self.mode = mode
        self.scale = scale
        self.lam = lam
        self.clamp = clamp
        if u0 is not None and not torch.is_tensor(u0):
            u0 = torch.tensor(u0, dtype=torch.get_default_dtype())
        self.u0_const = u0
        if mode == "decreasing":
            # start near S = S0: softplus(-4) = 0.018, so S barely declines until
            # the data asks it to, rather than collapsing in the first days.
            with torch.no_grad():
                for m_ in reversed(list(self.net.mlp.net)):
                    if isinstance(m_, nn.Linear):
                        m_.bias.fill_(-4.0); break

    def ramp(self, t):
        return 1.0 - torch.exp(-t / self.lam)

    def forward(self, z, t, u0=None):
        g = self.net(z)
        r = self.ramp(t)
        base = self.u0_const if u0 is None else u0
        if self.mode == "zero":
            return r * self.scale * nn.functional.softplus(g)
        if self.mode == "positive":
            return base * torch.exp(r * torch.clamp(g, -self.clamp, self.clamp))
        if self.mode == "decreasing":
            return base * torch.exp(-r * nn.functional.softplus(g))
        raise ValueError(self.mode)


def d_dx(y, x):
    return torch.autograd.grad(y, x, torch.ones_like(y), create_graph=True)[0]


# --------------------------------------------------------------------------
# Single-location (city) PINN:  S(t), E(t,a), I(t,a), P(t,b),  D = 0
# --------------------------------------------------------------------------
class CityPINN(nn.Module):
    """The age-structured SEIP system at one governorate with diffusion off.

    With D = 0 the spatial operator vanishes and the system decouples into an
    initial-boundary-value problem in (t, a) and (t, b). That is the reduction
    this project's per-region work already uses, and it makes I(t, a) a genuine
    two-dimensional surface -- the object the age structure actually adds, and
    the one the two architectures can be compared on.
    """

    def __init__(self, kind="ff", width=64, depth=4, M=4, K_t=8, K_a=4, K_b=6,
                 scaleI=1e-5, scaleP=1e-4, seed=0, hard_ic=True,
                 S0=1.0, I0=1e-5, P0=1e-4):
        super().__init__()
        torch.manual_seed(seed)
        self.kind = kind
        # The two design factors are independent and must be set independently:
        # an earlier version keyed both off a single flag, which confounded the
        # state embedding with the beta parameterisation and made the DPINN vs
        # FF-PINN difference unattributable to either.
        state_ff, beta_kind = ARCH[kind]
        self.state_ff, self.beta_kind = state_ff, beta_kind
        ff2 = [K_t, K_a] if state_ff else None
        ff2b = [K_t, K_b] if state_ff else None
        ff1 = [K_t] if state_ff else None
        self.hard_ic = hard_ic
        if hard_ic:
            # S0, I0 and P0 are supplied at build time so the IC is exact.
            self._S = HardIC(RawNet(1, width, depth, ff1), "decreasing", u0=S0)
            self._E = HardIC(RawNet(2, width, depth, ff2), "zero", scale=scaleI)
            self._I = HardIC(RawNet(2, width, depth, ff2), "positive", u0=I0)
            self._P = HardIC(RawNet(2, width, depth, ff2b), "positive", u0=P0)
        else:
            self._S = FieldNet(1, width, depth, ff1, scale=1.0)
            self._E = FieldNet(2, width, depth, ff2, scale=scaleI)
            self._I = FieldNet(2, width, depth, ff2, scale=scaleI)
            self._P = FieldNet(2, width, depth, ff2b, scale=scaleP)
        self.beta = (BetaNetFourier(M=M) if beta_kind == "fourier" else BetaNetMLP())

        aq, wq = age_nodes()
        bq, vq = path_nodes()
        self.register_buffer("aq", torch.tensor(aq))
        self.register_buffer("wf", torch.tensor(wq * seip.f_kernel(aq)))
        self.register_buffer("wsig", torch.tensor(wq * seip.SIGMA0))
        self.register_buffer("walpha", torch.tensor(wq * seip.ALPHA0 * seip.f_kernel(aq)))
        self.register_buffer("bq", torch.tensor(bq))
        self.register_buffer("wh", torch.tensor(vq * seip.h_kernel(bq)))

    # -- coordinate normalization -----------------------------------------
    @staticmethod
    def nt(t):
        return 2.0 * t / seip.T_MAX - 1.0

    @staticmethod
    def na(a):
        return 2.0 * a / seip.A_MAX - 1.0

    @staticmethod
    def nb(b):
        return 2.0 * b / seip.B_MAX - 1.0

    # -- field evaluation --------------------------------------------------
    def S_of(self, t):
        z = self.nt(t).reshape(-1, 1)
        return self._S(z, t) if self.hard_ic else self._S(z)

    def E(self, z, t):
        return self._E(z, t) if self.hard_ic else self._E(z)

    def I(self, z, t):
        return self._I(z, t) if self.hard_ic else self._I(z)

    def P(self, z, t):
        return self._P(z, t) if self.hard_ic else self._P(z)

    def _grid(self, t, q):
        """Outer product of times t (n,) and nodes q (m,) -> flat (n*m,) pairs."""
        n, m = t.numel(), q.numel()
        return (t.reshape(-1, 1).expand(n, m).reshape(-1),
                q.reshape(1, -1).expand(n, m).reshape(-1), n, m)

    def EIP_on_nodes(self, t):
        """E, I on the a-quadrature nodes and P on the b-nodes, for times t."""
        ta, aa, n, ma = self._grid(t, self.aq)
        za = torch.stack([self.nt(ta), self.na(aa)], 1)
        Emat = self.E(za, ta).reshape(n, ma)
        Imat = self.I(za, ta).reshape(n, ma)
        tb, bb, _, mb = self._grid(t, self.bq)
        zb = torch.stack([self.nt(tb), self.nb(bb)], 1)
        Pmat = self.P(zb, tb).reshape(n, mb)
        return Emat, Imat, Pmat

    def lambdas(self, t, Imat=None, Pmat=None):
        if Imat is None:
            _, Imat, Pmat = self.EIP_on_nodes(t)
        b10, b20 = self.beta(self.nt(t).reshape(-1, 1))
        lamI = b10 * (Imat @ self.wf)
        lamP = b20 * (Pmat @ self.wh)
        return lamI, lamP, b10, b20

    def IP_on_nodes(self, t):
        """I on the a-nodes and P on the b-nodes only. The observation operator
        does not involve E, and evaluating it there was a third of the cost."""
        ta, aa, n, ma = self._grid(t, self.aq)
        Imat = self.I(torch.stack([self.nt(ta), self.na(aa)], 1), ta).reshape(n, ma)
        tb, bb, _, mb = self._grid(t, self.bq)
        Pmat = self.P(torch.stack([self.nt(tb), self.nb(bb)], 1), tb).reshape(n, mb)
        return Imat, Pmat

    def observable(self, t):
        """H(t) = S(t) [ int b1 I da + int b2 P db ] -- what a bulletin measures."""
        Imat, Pmat = self.IP_on_nodes(t)
        b10, b20 = self.beta(self.nt(t).reshape(-1, 1))
        return self.S_of(t) * (b10 * (Imat @ self.wf) + b20 * (Pmat @ self.wh))

    # -- residuals ---------------------------------------------------------
    def residuals(self, t_col):
        t = t_col.clone().requires_grad_(True)

        # S: ordinary differential residual
        S = self.S_of(t)
        dS = d_dx(S, t)
        ta, aa, n, ma = self._grid(t, self.aq)
        ta = ta.clone().requires_grad_(True); aa = aa.clone().requires_grad_(True)
        za = torch.stack([self.nt(ta), self.na(aa)], 1)
        Ef = self.E(za, ta); If = self.I(za, ta)
        Emat, Imat = Ef.reshape(n, ma), If.reshape(n, ma)

        tb, bb, _, mb = self._grid(t, self.bq)
        tb = tb.clone().requires_grad_(True); bb = bb.clone().requires_grad_(True)
        zb = torch.stack([self.nt(tb), self.nb(bb)], 1)
        Pf = self.P(zb, tb); Pmat = Pf.reshape(n, mb)

        b10, b20 = self.beta(self.nt(t).reshape(-1, 1))
        lamI = b10 * (Imat @ self.wf)
        lamP = b20 * (Pmat @ self.wh)
        foi = lamI + lamP

        r_S = dS + S * foi - seip.LAMBDA_REC + seip.MU1 * S
        r_E = d_dx(Ef, ta) + d_dx(Ef, aa) + seip.R_E * Ef
        r_I = d_dx(If, ta) + d_dx(If, aa) + seip.R_I * If
        r_P = d_dx(Pf, tb) + d_dx(Pf, bb) + seip.MU4 * Pf

        # renewal conditions -- the non-local boundary conditions at a = 0, b = 0,
        # which is where the unknowns enter the system at all
        z0 = torch.stack([self.nt(t), torch.full_like(t, -1.0)], 1)
        E0 = self.E(z0, t); I0 = self.I(z0, t); P0 = self.P(z0, t)
        r_E0 = E0 - S * foi
        r_I0 = I0 - (Emat @ self.wsig)
        r_P0 = P0 - (Imat @ self.walpha)
        return r_S, r_E, r_I, r_P, r_E0, r_I0, r_P0


# --------------------------------------------------------------------------
# Data term
# --------------------------------------------------------------------------
class CityData:
    """Interval-averaged observation operator and the two data objectives."""

    def __init__(self, panel, gi, n_q=2, objective="poisson"):
        m = panel.gov == gi
        self.t0 = torch.tensor(panel.t0[m]); self.t1 = torch.tensor(panel.t1[m])
        self.y = torch.tensor(panel.y[m]); self.c = torch.tensor(panel.dC[m])
        self.w = torch.tensor(panel.w[m])
        self.N = float(panel.pop[gi])
        self.name = panel.names[gi]
        self.objective = objective
        x, w = gauss_legendre(-1.0, 1.0, n_q)
        self.qx = torch.tensor(x); self.qw = torch.tensor(w) / 2.0
        self.n_obs = int(m.sum())
        self.Ndt = self.N * (self.t1 - self.t0)
        self.dev_scale = 1.0
        self.yscale = float(np.sqrt(np.mean((panel.w[m] * panel.y[m]) ** 2)))

    def eval_times(self):
        mid = 0.5 * (self.t0 + self.t1); half = 0.5 * (self.t1 - self.t0)
        return (mid.reshape(-1, 1) + half.reshape(-1, 1) * self.qx.reshape(1, -1)).reshape(-1)

    def average(self, H):
        return (H.reshape(self.n_obs, -1) * self.qw.reshape(1, -1)).sum(1)

    def residual(self, yhat):
        if self.objective == "wls":
            return self.w * (yhat - self.y) / self.yscale
        lam = torch.clamp(yhat * self.Ndt, min=1e-10)
        c = self.c
        term = torch.where(c > 0, c * torch.log(torch.clamp(c, min=1e-300) / lam),
                           torch.zeros_like(c))
        d2 = torch.clamp(2.0 * (term - (c - lam)), min=0.0)
        return torch.sign(c - lam) * torch.sqrt(d2) / self.dev_scale


# --------------------------------------------------------------------------
# Training
# --------------------------------------------------------------------------
def city_initial_condition(panel, gi, grid, **ic_opts):
    """The same data-anchored IC the deterministic arm uses, at one location."""
    ic = seip.build_initial_condition(seip.Grid(dt=grid.dt, nx=1),
                                      np.array([0.0]),
                                      np.array([panel.y_first[gi]]),
                                      np.array([panel.cum_first[gi]]), **ic_opts)
    return (float(ic.S0[0]), ic.I0[:, 0].copy(), ic.P0[:, 0].copy(),
            float(ic.I_agg_nodal[0]), float(ic.P_agg_nodal[0]), ic.opts)


def train_city(model: CityPINN, data: CityData, S0, Iagg0, Pagg0,
               n_col=192, epochs=3000, lr=3e-3, w_pde=1.0, w_ren=20.0, w_ic=50.0,
               w_smooth=3e-2, lbfgs=250, log_every=500, verbose=True,
               pretrain=800):
    """Staged: (1) fit the fields to the initial condition and the renewal
    relations with beta frozen, (2) release everything under Adam, (3) refine
    with L-BFGS. Starting the joint fit cold lets the two networks trade against
    each other before either means anything.

    w_ren is deliberately large. The renewal condition E(t,0) = S(t)[int b1 I +
    int b2 P] is the ONLY relation in which beta appears at all, so it is the
    only thing tying BetaNet to FieldNet. Under-weight it and the two networks
    decouple: the data term constrains their product and never the factors, and
    the optimizer can cheapen the loss by inflating beta instead of growing I and
    P. That is precisely the dual-network failure this project's per-region work
    diagnosed, so the coupling weight is reported, not left at a default."""
    T = seip.T_MAX
    aq_t = torch.tensor(np.linspace(0, seip.A_MAX, 33))
    bq_t = torch.tensor(np.linspace(0, seip.B_MAX, 41))
    I0_flat = Iagg0 / seip.A_MAX
    P0_flat = Pagg0 / seip.B_MAX

    def ic_loss():
        if getattr(model, "hard_ic", False):
            # exact by construction; the penalty would only add a zero term
            return torch.zeros((), dtype=torch.get_default_dtype())
        z = torch.zeros(1)
        s = model.S_of(z)
        za = torch.stack([model.nt(torch.zeros_like(aq_t)), model.na(aq_t)], 1)
        zb = torch.stack([model.nt(torch.zeros_like(bq_t)), model.nb(bq_t)], 1)
        E0 = model.E(za); I0 = model.I(za); P0 = model.P(zb)
        sc_I = max(I0_flat, 1e-12); sc_P = max(P0_flat, 1e-12)
        return (((s - S0) / max(S0, 1e-9)) ** 2).mean() \
            + ((E0 / sc_I) ** 2).mean() \
            + (((I0 - I0_flat) / sc_I) ** 2).mean() \
            + (((P0 - P0_flat) / sc_P) ** 2).mean()

    hist = []

    def physics(n):
        tc = torch.rand(n) * T
        tc = torch.cat([tc, torch.rand(n // 4) * 14.0])      # oversample near t=0
        rS, rE, rI, rP, rE0, rI0, rP0 = model.residuals(tc)
        sI = max(Iagg0 / seip.A_MAX, 1e-12); sP = max(Pagg0 / seip.B_MAX, 1e-12)
        pde = (rS ** 2).mean() + ((rE / sI) ** 2).mean() + ((rI / sI) ** 2).mean() \
            + ((rP / sP) ** 2).mean()
        ren = ((rE0 / sI) ** 2).mean() + ((rI0 / sI) ** 2).mean() \
            + ((rP0 / sP) ** 2).mean()
        return pde, ren

    def smooth():
        tt = torch.linspace(0, T, 200).reshape(-1, 1).requires_grad_(True)
        b10, b20 = model.beta(model.nt(tt))
        d1 = d_dx(b10, tt); d2 = d_dx(b20, tt)
        return ((d1 / torch.clamp(b10.mean(), min=1e-300)) ** 2).mean() \
            + ((d2 / torch.clamp(b20.mean(), min=1e-300)) ** 2).mean()

    # ---- stage 1: fields to the IC and the renewal relations ----------------
    for p in model.beta.parameters():
        p.requires_grad_(False)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    for it in range(pretrain):
        opt.zero_grad()
        pde, ren = physics(n_col // 2)
        loss = w_ic * ic_loss() + 0.1 * pde + w_ren * ren
        loss.backward(); opt.step()
    for p in model.beta.parameters():
        p.requires_grad_(True)

    # ---- stage 2: joint Adam ------------------------------------------------
    te = data.eval_times()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr / 50)
    for it in range(epochs):
        opt.zero_grad()
        yhat = data.average(model.observable(te))
        r = data.residual(yhat)
        dloss = (r ** 2).mean()
        pde, ren = physics(n_col)
        loss = dloss + w_pde * pde + w_ren * ren + w_ic * ic_loss() + w_smooth * smooth()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        opt.step(); sched.step()
        if verbose and (it % log_every == 0 or it == epochs - 1):
            print(f"    adam {it:5d}  total {loss.item():.4e}  data {dloss.item():.4e}  "
                  f"pde {pde.item():.3e}  renewal {ren.item():.3e}")
        hist.append(float(loss.item()))

    # ---- stage 3: L-BFGS ----------------------------------------------------
    if lbfgs:
        optL = torch.optim.LBFGS(model.parameters(), max_iter=lbfgs, history_size=50,
                                 tolerance_grad=1e-12, tolerance_change=1e-14,
                                 line_search_fn="strong_wolfe")
        tc_fixed = torch.linspace(0.0, T, n_col)

        def closure():
            optL.zero_grad()
            yhat = data.average(model.observable(te))
            r = data.residual(yhat)
            rS, rE, rI, rP, rE0, rI0, rP0 = model.residuals(tc_fixed)
            sI = max(Iagg0 / seip.A_MAX, 1e-12); sP = max(Pagg0 / seip.B_MAX, 1e-12)
            pde = (rS ** 2).mean() + ((rE / sI) ** 2).mean() + ((rI / sI) ** 2).mean() \
                + ((rP / sP) ** 2).mean()
            ren = ((rE0 / sI) ** 2).mean() + ((rI0 / sI) ** 2).mean() \
                + ((rP0 / sP) ** 2).mean()
            loss = (r ** 2).mean() + w_pde * pde + w_ren * ren + w_ic * ic_loss() \
                + w_smooth * smooth()
            loss.backward()
            return loss

        try:
            optL.step(closure)
        except RuntimeError as e:
            if verbose:
                print("    L-BFGS stopped:", e)
    return hist


@torch.no_grad()
def city_predict(model: CityPINN, t_grid, a_grid, b_grid):
    """Everything a figure needs, on a regular grid."""
    t = torch.tensor(np.asarray(t_grid, float))
    S = model.S_of(t).numpy()
    b10, b20 = model.beta(model.nt(t).reshape(-1, 1))
    Emat, Imat, Pmat = model.EIP_on_nodes(t)
    lamI = b10 * (Imat @ model.wf); lamP = b20 * (Pmat @ model.wh)
    H = (model.S_of(t) * (lamI + lamP)).numpy()

    def surf(net, tg, qg, norm):
        tt = torch.tensor(np.repeat(np.asarray(tg, float), len(qg)))
        qq = torch.tensor(np.tile(np.asarray(qg, float), len(tg)))
        z = torch.stack([model.nt(tt), norm(qq)], 1)
        return net(z, tt).reshape(len(tg), len(qg)).numpy()

    return dict(S=S, b10=b10.numpy(), b20=b20.numpy(), H=H,
                I_surface=surf(model.I, t_grid, a_grid, model.na),
                E_surface=surf(model.E, t_grid, a_grid, model.na),
                P_surface=surf(model.P, t_grid, b_grid, model.nb),
                I_agg=(Imat @ torch.tensor(age_nodes()[1])).numpy())


def R0_from(model, t_grid, kappa, J1, J2):
    with torch.no_grad():
        t = torch.tensor(np.asarray(t_grid, float))
        S = model.S_of(t).numpy()
        b10, b20 = model.beta(model.nt(t).reshape(-1, 1))
    return seip.R0_of_t(b10.numpy(), b20.numpy(), S, kappa, J1, J2)


# --------------------------------------------------------------------------
# Full-domain PINN:  S(t,x), E(t,a,x), I(t,a,x), P(t,b,x)
# --------------------------------------------------------------------------
class DomainPINN(nn.Module):
    """The full age-structured transport-diffusion system on [0, X_max].

    Same dual-network design as CityPINN; the fields now carry a spatial
    coordinate, so the PDE residuals pick up D d2/dx2 and the homogeneous
    Neumann conditions are enforced as a soft penalty at the two ends.
    """

    def __init__(self, kind="ff", width=64, depth=4, M=4,
                 K_t=8, K_x=4, K_a=4, K_b=6, scaleI=1e-5, scaleP=1e-4, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.kind = kind
        state_ff, beta_kind = ARCH[kind]
        self.state_ff, self.beta_kind = state_ff, beta_kind
        ff_tx = [K_t, K_x] if state_ff else None
        ff_tax = [K_t, K_a, K_x] if state_ff else None
        ff_tbx = [K_t, K_b, K_x] if state_ff else None
        self.S = FieldNet(2, width, depth, ff_tx, scale=1.0)
        self.E = FieldNet(3, width, depth, ff_tax, scale=scaleI)
        self.I = FieldNet(3, width, depth, ff_tax, scale=scaleI)
        self.P = FieldNet(3, width, depth, ff_tbx, scale=scaleP)
        self.beta = (BetaNetFourier(M=M) if beta_kind == "fourier" else BetaNetMLP())
        aq, wq = age_nodes(); bq, vq = path_nodes()
        self.register_buffer("aq", torch.tensor(aq))
        self.register_buffer("wf", torch.tensor(wq * seip.f_kernel(aq)))
        self.register_buffer("wsig", torch.tensor(wq * seip.SIGMA0))
        self.register_buffer("walpha", torch.tensor(wq * seip.ALPHA0 * seip.f_kernel(aq)))
        self.register_buffer("bq", torch.tensor(bq))
        self.register_buffer("wh", torch.tensor(vq * seip.h_kernel(bq)))

    def __repr_name__(self):
        return self.kind

    @staticmethod
    def nt(t):
        return 2.0 * t / seip.T_MAX - 1.0

    @staticmethod
    def na(a):
        return 2.0 * a / seip.A_MAX - 1.0

    @staticmethod
    def nb(b):
        return 2.0 * b / seip.B_MAX - 1.0

    def nx(self, x):
        return 2.0 * x / self.x_max - 1.0

    def set_domain(self, x_max):
        self.x_max = float(x_max)

    def S_of(self, t, x):
        return self.S(torch.stack([self.nt(t), self.nx(x)], 1))

    def _tile(self, t, x, q):
        n, m = t.numel(), q.numel()
        tt = t.reshape(-1, 1).expand(n, m).reshape(-1)
        xx = x.reshape(-1, 1).expand(n, m).reshape(-1)
        qq = q.reshape(1, -1).expand(n, m).reshape(-1)
        return tt, xx, qq, n, m

    def IP_on_nodes(self, t, x):
        tt, xx, aa, n, ma = self._tile(t, x, self.aq)
        Imat = self.I(torch.stack([self.nt(tt), self.na(aa), self.nx(xx)], 1)).reshape(n, ma)
        tb, xb, bb, _, mb = self._tile(t, x, self.bq)
        Pmat = self.P(torch.stack([self.nt(tb), self.nb(bb), self.nx(xb)], 1)).reshape(n, mb)
        return Imat, Pmat

    def observable(self, t, x):
        Imat, Pmat = self.IP_on_nodes(t, x)
        b10, b20 = self.beta(self.nt(t).reshape(-1, 1))
        return self.S_of(t, x) * (b10 * (Imat @ self.wf) + b20 * (Pmat @ self.wh))

    def residuals(self, t_col, x_col, diff: seip.Diffusion):
        D1, D2, D3, D4 = diff.as_tuple()
        sx = 2.0 / self.x_max                      # d/dx = sx * d/dxi
        t = t_col.clone().requires_grad_(True)
        x = x_col.clone().requires_grad_(True)

        S = self.S_of(t, x)
        dSt = d_dx(S, t)
        dSx = d_dx(S, x)
        dSxx = d_dx(dSx, x)

        tt, xx, aa, n, ma = self._tile(t, x, self.aq)
        tt = tt.clone().requires_grad_(True); xx = xx.clone().requires_grad_(True)
        aa = aa.clone().requires_grad_(True)
        za = torch.stack([self.nt(tt), self.na(aa), self.nx(xx)], 1)
        Ef = self.E(za); If = self.I(za)
        Emat, Imat = Ef.reshape(n, ma), If.reshape(n, ma)

        tb, xb, bb, _, mb = self._tile(t, x, self.bq)
        tb = tb.clone().requires_grad_(True); xb = xb.clone().requires_grad_(True)
        bb = bb.clone().requires_grad_(True)
        zb = torch.stack([self.nt(tb), self.nb(bb), self.nx(xb)], 1)
        Pf = self.P(zb); Pmat = Pf.reshape(n, mb)

        b10, b20 = self.beta(self.nt(t).reshape(-1, 1))
        foi = b10 * (Imat @ self.wf) + b20 * (Pmat @ self.wh)

        def lap(f, v):
            g1 = d_dx(f, v)
            return d_dx(g1, v)

        r_S = dSt - D1 * dSxx + S * foi - seip.LAMBDA_REC + seip.MU1 * S
        r_E = d_dx(Ef, tt) + d_dx(Ef, aa) - D2 * lap(Ef, xx) + seip.R_E * Ef
        r_I = d_dx(If, tt) + d_dx(If, aa) - D3 * lap(If, xx) + seip.R_I * If
        r_P = d_dx(Pf, tb) + d_dx(Pf, bb) - D4 * lap(Pf, xb) + seip.MU4 * Pf

        z0 = torch.stack([self.nt(t), torch.full_like(t, -1.0), self.nx(x)], 1)
        r_E0 = self.E(z0) - S * foi
        r_I0 = self.I(z0) - (Emat @ self.wsig)
        r_P0 = self.P(z0) - (Imat @ self.walpha)
        return r_S, r_E, r_I, r_P, r_E0, r_I0, r_P0

    def neumann(self, t_b):
        """Homogeneous Neumann on all four fields at x = 0 and x = X_max."""
        out = []
        for xv in (0.0, self.x_max):
            x = torch.full_like(t_b, xv).requires_grad_(True)
            S = self.S_of(t_b, x)
            out.append(d_dx(S, x))
            for net, q, norm in ((self.E, self.aq, self.na), (self.I, self.aq, self.na),
                                 (self.P, self.bq, self.nb)):
                tt, xx, qq, n, m = self._tile(t_b, x, q)
                xx = xx.clone().requires_grad_(True)
                f = net(torch.stack([self.nt(tt), norm(qq), self.nx(xx)], 1))
                out.append(d_dx(f, xx))
        return torch.cat(out)

In [ ]:
%%writefile fit_one.py
#!/usr/bin/env python3
"""fit_one.py -- one (city, arm) fit on the REAL Yemen record, saved on its own.

Granularity is deliberate: each arm is a separate process writing its own
pickle, so a wall-clock limit can never discard a completed fit.  There is no
synthetic twin and no true parameter vector anywhere in this script; every
number it reports is a comparison of a fitted quantity against the OBSERVED
incidence, or between the two architectures.

    python fit_one.py --city "Al Hudaydah" --arm ff
"""
from __future__ import annotations
import argparse, copy, json, os, pickle, sys, time
import numpy as np, pandas as pd, torch
import seip, data_io, fit, pinn

torch.set_num_threads(2)
P = lambda *a: (print(*a, flush=True))


def slug(c): return c.lower().replace(" ", "_").replace("'", "")


def errors(pred, actual, label=""):
    """MAE, MSE, RMSE and two scale-free normalisations.

    MAE and RMSE are reported together because they answer different
    questions on this record: the incidence series is zero-inflated and
    spiky, so RMSE is dominated by the batched-reporting spikes while MAE
    tracks the typical interval.  Quoting one alone hides which it is.
    """
    pred = np.asarray(pred, float).ravel(); actual = np.asarray(actual, float).ravel()
    m = np.isfinite(pred) & np.isfinite(actual)
    pred, actual = pred[m], actual[m]
    e = pred - actual
    mae = float(np.mean(np.abs(e))); mse = float(np.mean(e ** 2))
    rmse = float(np.sqrt(mse))
    rms_a = float(np.sqrt(np.mean(actual ** 2)))
    rng = float(actual.max() - actual.min())
    return {f"{label}MAE": mae, f"{label}MSE": mse, f"{label}RMSE": rmse,
            f"{label}nRMSE_rms": rmse / rms_a if rms_a > 0 else np.nan,
            f"{label}nRMSE_range": rmse / rng if rng > 0 else np.nan,
            f"{label}n": int(pred.size)}


def city_panel(city, data_dir):
    panel = data_io.load_panel(data_dir, verbose=False)
    gi = panel.names.index(city)
    m = panel.gov == gi
    p = copy.copy(panel)
    p.gov = np.zeros(int(m.sum()), int)
    for a in ("t0", "t1", "y", "y_raw", "dC", "dC_raw", "w", "w_count", "w_rate"):
        if hasattr(panel, a):
            setattr(p, a, getattr(panel, a)[m].copy())
    p.names = [city]
    for a in ("xg", "pop", "y_first", "cum_first", "t_first"):
        setattr(p, a, np.asarray(getattr(panel, a))[[gi]].copy())
    p.xg = np.array([0.0])
    for a in ("true_zero", "first_hold"):
        v = getattr(panel, a, None)
        if v is not None:
            setattr(p, a, np.asarray(v)[[gi]].copy())
    # ng is a derived property on Panel; it follows from names/xg
    return p


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--city", required=True)
    ap.add_argument("--arm", required=True,
                    choices=["det", "dpinn", "dpinn_fs", "fdpinn", "ff"])
    ap.add_argument("--data-dir", default="data_real")
    ap.add_argument("--outdir", default="real_fits")
    ap.add_argument("--dt", type=float, default=0.25)
    ap.add_argument("--epochs", type=int, default=1000)
    ap.add_argument("--pretrain", type=int, default=800)
    ap.add_argument("--lbfgs", type=int, default=120)
    ap.add_argument("--n-col", type=int, default=192)
    ap.add_argument("--width", type=int, default=64)
    ap.add_argument("--depth", type=int, default=4)
    ap.add_argument("--M", type=int, default=4)
    ap.add_argument("--w-ren", type=float, default=20.0)
    ap.add_argument("--w-pde", type=float, default=1.0)
    ap.add_argument("--w-ic", type=float, default=50.0)
    ap.add_argument("--w-smooth", type=float, default=3e-2)
    ap.add_argument("--objective", default="poisson", choices=["poisson", "wls"])
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()

    t_all = time.time()
    grid = seip.Grid(dt=a.dt, nx=41)
    grid1 = seip.Grid(dt=a.dt, nx=1)
    panel = city_panel(a.city, a.data_dir)
    panel.set_weights("count")
    kap, J1, J2 = seip.ngm_integrals()
    nz = int((panel.y > 0).sum())
    P(f"[{a.city} | {a.arm}] rows {len(panel.y)}  nonzero {nz} "
      f"({100*nz/len(panel.y):.1f}%)  pop {panel.pop[0]:,.0f}  "
      f"mean y {panel.y.mean():.4e}  peak y {panel.y.max():.4e}")

    S0, _, _, Iagg0, Pagg0, icopt = pinn.city_initial_condition(panel, 0, grid)
    P(f"  IC  S0={S0:.6f}  I_agg0={Iagg0:.4e}  P_agg0={Pagg0:.4e}  "
      f"tau_I={icopt['tau_I']:.4f} d")

    os.makedirs(os.path.join(a.outdir, slug(a.city)), exist_ok=True)
    out = os.path.join(a.outdir, slug(a.city), f"{a.arm}.pkl")

    if a.arm == "det":
        prob = fit.Problem(grid1, panel, objective=a.objective)
        _, _, _, seed = fit.grid_search(prob, seip.Diffusion(0, 0, 0, 0), n=11,
                                        verbose=False)
        res = fit.fit(prob, seip.Diffusion(0, 0, 0, 0), M=a.M, seed_beta=seed,
                      n_jobs=2, max_nfev=200, label="deterministic", verbose=False)
        _, Sb, rec = seip.solve_forward(grid1, prob.ic, res.beta10_t, res.beta20_t,
                                        seip.Diffusion(0, 0, 0, 0), record_full=True)
        pack = dict(name="Deterministic (Poisson deviance)", arm="det",
                    beta10=res.beta10_t, beta20=res.beta20_t, R0=res.R0_t,
                    S=Sb, I_surface=rec["I_full"][:, :, 0],
                    P_surface=rec["P_full"][:, :, 0], yhat=res.yhat, t=grid1.t,
                    env_share=float(seip.route_share(res.beta10_t, res.beta20_t,
                                                     J2, grid1)[0]),
                    converged=bool(getattr(res, "converged", True)))
    else:
        nice = pinn.ARCH_NAME[a.arm]
        model = pinn.CityPINN(kind=a.arm,
                              width=a.width, depth=a.depth, M=a.M,
                              scaleI=max(Iagg0 / seip.A_MAX, 1e-9),
                              scaleP=max(Pagg0 / seip.B_MAX, 1e-9), seed=a.seed,
                              hard_ic=True, S0=S0,
                              I0=max(Iagg0 / seip.A_MAX, 1e-12),
                              P0=max(Pagg0 / seip.B_MAX, 1e-12))
        data = pinn.CityData(panel, 0, objective=a.objective)
        hist = pinn.train_city(model, data, S0, Iagg0, Pagg0, n_col=a.n_col,
                               epochs=a.epochs, pretrain=a.pretrain, lbfgs=a.lbfgs,
                               log_every=max(a.epochs // 5, 1), w_ren=a.w_ren,
                               w_pde=a.w_pde, w_ic=a.w_ic, w_smooth=a.w_smooth)
        pred = pinn.city_predict(model, grid1.t, grid1.a, grid1.b)
        R0 = pinn.R0_from(model, grid1.t, kap, J1, J2)
        with torch.no_grad():
            yhat = data.average(model.observable(data.eval_times())).numpy()
        pack = dict(name=nice, arm=a.arm, beta10=pred["b10"], beta20=pred["b20"],
                    R0=R0, S=pred["S"], H=pred["H"], I_surface=pred["I_surface"],
                    P_surface=pred["P_surface"], yhat=yhat, hist=hist, t=grid1.t,
                    env_share=float(seip.route_share(pred["b10"], pred["b20"],
                                                     J2, grid1)[0]),
                    n_params=int(sum(p.numel() for p in model.parameters())))

    pack["seconds"] = time.time() - t_all
    pack["city"] = a.city
    row = dict(city=a.city, method=pack["name"], arm=a.arm)
    row.update(errors(pack["yhat"], panel.y, ""))            # against OBSERVED data
    row["R0_mean"] = float(pack["R0"].mean())
    row["R0_min"] = float(pack["R0"].min())
    row["R0_max"] = float(pack["R0"].max())
    row["R0_final"] = float(pack["R0"][-1])
    row["frac_t_R0_gt1"] = float(np.mean(pack["R0"] > 1.0))
    row["env_share"] = pack["env_share"]
    row["seconds"] = pack["seconds"]
    pack["row"] = row
    pack["observed"] = dict(t0=panel.t0, t1=panel.t1, y=panel.y, pop=float(panel.pop[0]))

    with open(out, "wb") as f:
        pickle.dump(pack, f)
    P(f"  {pack['name']}: {pack['seconds']:.0f}s   "
      f"MAE {row['MAE']:.4e}  RMSE {row['RMSE']:.4e}  "
      f"R0 {row['R0_min']:.2f}-{row['R0_max']:.2f} (mean {row['R0_mean']:.2f}, "
      f"{100*row['frac_t_R0_gt1']:.0f}% of days >1)  env share {row['env_share']:.3f}")
    P(f"  -> {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile holdout.py
#!/usr/bin/env python3
"""holdout.py -- forward out-of-sample validation for one (city, arm).

Every error reported in the main study is training error, and the FF-PINN
carries capacity the other arms do not: its states are free network outputs
tied to the PDE only softly, so it can represent curves the PDE itself cannot
produce.  That is the same property that leaves its R0 unconstrained.  So the
in-sample advantage has to be tested, not assumed to be reconstruction.

Protocol: fit on t <= TRAIN_UNTIL, predict the whole window, score the two
segments separately.  The initial condition is anchored on the first bulletin,
which lies inside the training segment, so no held-out information enters the
fit through the IC.

    python holdout.py --city "Al Hudaydah" --arm ff --train-until 200
"""
from __future__ import annotations
import argparse, copy, os, pickle, time
import numpy as np, torch
import seip, data_io, fit, pinn, fit_one

torch.set_num_threads(2)
P = lambda *a: print(*a, flush=True)

ROW_ATTRS = ("t0", "t1", "y", "y_raw", "dC", "dC_raw", "w", "w_count", "w_rate", "gov")


def time_slice(panel, keep):
    """Rows of a single-location panel selected by a boolean mask."""
    p = copy.copy(panel)
    for a in ROW_ATTRS:
        if hasattr(panel, a) and getattr(panel, a) is not None:
            v = np.asarray(getattr(panel, a))
            if v.shape[:1] == keep.shape:
                setattr(p, a, v[keep].copy())
    return p


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--city", required=True)
    ap.add_argument("--arm", required=True,
                    choices=["det", "dpinn", "dpinn_fs", "fdpinn", "ff"])
    ap.add_argument("--train-until", type=float, default=200.0)
    ap.add_argument("--mode", default="forward", choices=["forward", "interleave"],
                    help="forward: fit on t<=train_until and predict beyond it, "
                         "which tests FORECASTING and is dominated by beta being "
                         "extrapolated outside its data support. interleave: hold "
                         "out every 4th interval, spread across the whole window, "
                         "so beta interpolates everywhere and the test isolates "
                         "whether the fitted surface generalises to unseen "
                         "observation times.")
    ap.add_argument("--data-dir", default="data_real")
    ap.add_argument("--outdir", default="holdout_fits")
    ap.add_argument("--dt", type=float, default=0.25)
    ap.add_argument("--epochs", type=int, default=1000)
    ap.add_argument("--pretrain", type=int, default=800)
    ap.add_argument("--lbfgs", type=int, default=120)
    ap.add_argument("--n-col", type=int, default=192)
    ap.add_argument("--M", type=int, default=4)
    ap.add_argument("--w-pde", type=float, default=1.0)
    ap.add_argument("--w-ren", type=float, default=20.0)
    ap.add_argument("--w-ic", type=float, default=50.0)
    ap.add_argument("--w-smooth", type=float, default=3e-2)
    ap.add_argument("--objective", default="poisson", choices=["poisson", "wls"])
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--tag", default="")
    a = ap.parse_args()

    t_all = time.time()
    grid1 = seip.Grid(dt=a.dt, nx=1)
    full = fit_one.city_panel(a.city, a.data_dir)
    full.set_weights("count")
    if a.mode == "forward":
        TR = full.t1 <= a.train_until
        TE = full.t0 >= a.train_until
        desc = f"train t<={a.train_until:.0f} (forecast test)"
    else:
        idx = np.arange(len(full.y))
        TE = (idx % 4) == 2          # index 0 stays in train: it anchors the IC
        TR = ~TE
        desc = "every 4th interval held out (generalisation test)"
    train = time_slice(full, TR)
    train.set_weights("count")
    P(f"[{a.city} | {a.arm} | {desc}] "
      f"{len(full.y)} intervals -> train {int(TR.sum())}, held out {int(TE.sum())}, "
      f"neither {int((~TR & ~TE).sum())}")

    S0, _, _, Iagg0, Pagg0, icopt = pinn.city_initial_condition(train, 0, grid1)
    P(f"  IC from the training segment: S0={S0:.6f}  I_agg0={Iagg0:.4e}")

    if a.arm == "det":
        prob_tr = fit.Problem(grid1, train, objective=a.objective)
        _, _, _, seed = fit.grid_search(prob_tr, seip.Diffusion(0, 0, 0, 0), n=11,
                                        verbose=False)
        res = fit.fit(prob_tr, seip.Diffusion(0, 0, 0, 0), M=a.M, seed_beta=seed,
                      n_jobs=2, max_nfev=200, label="det-holdout", verbose=False)
        b10, b20, R0 = res.beta10_t, res.beta20_t, res.R0_t
        # score on EVERY interval, with the betas fitted on the training segment
        prob_full = fit.Problem(grid1, full, objective=a.objective)
        _, _, yhat_full, _ = prob_full.predict(b10, b20, seip.Diffusion(0, 0, 0, 0))
        name = "Deterministic (Poisson deviance)"
    else:
        name = pinn.ARCH_NAME[a.arm]
        model = pinn.CityPINN(kind=a.arm,
                              width=64, depth=4, M=a.M,
                              scaleI=max(Iagg0 / seip.A_MAX, 1e-9),
                              scaleP=max(Pagg0 / seip.B_MAX, 1e-9), seed=a.seed,
                              hard_ic=True, S0=S0,
                              I0=max(Iagg0 / seip.A_MAX, 1e-12),
                              P0=max(Pagg0 / seip.B_MAX, 1e-12))
        d_tr = pinn.CityData(train, 0, objective=a.objective)
        pinn.train_city(model, d_tr, S0, Iagg0, Pagg0, n_col=a.n_col,
                        epochs=a.epochs, pretrain=a.pretrain, lbfgs=a.lbfgs,
                        log_every=max(a.epochs // 4, 1), w_ren=a.w_ren,
                        w_pde=a.w_pde, w_ic=a.w_ic, w_smooth=a.w_smooth)
        d_full = pinn.CityData(full, 0, objective=a.objective)
        with torch.no_grad():
            yhat_full = d_full.average(model.observable(d_full.eval_times())).numpy()
        kap, J1, J2 = seip.ngm_integrals()
        pred = pinn.city_predict(model, grid1.t, grid1.a, grid1.b)
        b10, b20 = pred["b10"], pred["b20"]
        R0 = pinn.R0_from(model, grid1.t, kap, J1, J2)

    row = dict(city=a.city, arm=a.arm, method=name, mode=a.mode,
               train_until=a.train_until, w_pde=a.w_pde, tag=a.tag,
               n_train=int(TR.sum()), n_test=int(TE.sum()))
    row.update(fit_one.errors(yhat_full[TR], full.y[TR], "in_"))
    row.update(fit_one.errors(yhat_full[TE], full.y[TE], "out_"))
    row["R0_mean"] = float(R0.mean()); row["R0_max"] = float(R0.max())
    row["R0_min"] = float(R0.min())
    row["env_share"] = float(seip.route_share(b10, b20,
                                              seip.ngm_integrals()[2], grid1)[0])
    row["seconds"] = time.time() - t_all

    os.makedirs(os.path.join(a.outdir, fit_one.slug(a.city)), exist_ok=True)
    out = os.path.join(a.outdir, fit_one.slug(a.city),
                       f"{a.arm}_{a.mode}{('_' + a.tag) if a.tag else ''}.pkl")
    with open(out, "wb") as f:
        pickle.dump(dict(row=row, beta10=b10, beta20=b20, R0=R0, t=grid1.t,
                         yhat=yhat_full, TR=TR, TE=TE,
                         observed=dict(t0=full.t0, t1=full.t1, y=full.y)), f)
    P(f"  {name}: {row['seconds']:.0f}s")
    P(f"    in-sample  MAE {row['in_MAE']:.4e}  RMSE {row['in_RMSE']:.4e}  "
      f"(n={row['n_train']})")
    P(f"    HELD OUT   MAE {row['out_MAE']:.4e}  RMSE {row['out_RMSE']:.4e}  "
      f"(n={row['n_test']})")
    P(f"    degradation out/in: MAE x{row['out_MAE']/row['in_MAE']:.2f}  "
      f"RMSE x{row['out_RMSE']/row['in_RMSE']:.2f}   "
      f"R0 {row['R0_min']:.2f}-{row['R0_max']:.2f}")
    P(f"  -> {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile plots_real.py
"""
plots_real.py -- publication figures for the real-record DPINN vs FF-PINN study.

Conventions inherited from plots.py: colourblind-safe Okabe-Ito accents, a
perceptually uniform ramp for fields, a grey-midpoint diverging ramp for signed
error, model and data compared on the SAME footing (interval averages over each
reported interval, which is what the loss sees), and nothing interpolated
between bulletins.

There is no synthetic twin here and no true parameter vector. Every error is a
fitted quantity against the OBSERVED incidence; every parameter panel is a
comparison BETWEEN architectures, never against a known answer.
"""
from __future__ import annotations

import os, pickle
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm, LogNorm
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

import seip, data_io

plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 170, "font.size": 9,
    "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": "white", "savefig.bbox": "tight",
})

ARMS = [("det", "Deterministic", "#666666", "-"),
        ("dpinn", "DPINN", "#0072B2", "--"),
        ("ff", "FF-PINN", "#D55E00", "-")]
C_INK, C_MUT = "#222222", "#666666"
FIELD_CMAP = "viridis"
DIVERGE = LinearSegmentedColormap.from_list(
    "bwr_grey", ["#2166AC", "#7FA8CE", "#D9D9D9", "#E8977A", "#B2182B"])


def slug(c): return c.lower().replace(" ", "_").replace("'", "")


def load(cities, root="real_fits"):
    R = {}
    for c in cities:
        R[c] = {}
        for arm, *_ in ARMS:
            p = os.path.join(root, slug(c), f"{arm}.pkl")
            if os.path.exists(p):
                with open(p, "rb") as f:
                    R[c][arm] = pickle.load(f)
    return R


def _step(ax, t0, t1, y, **kw):
    """Draw each reported interval across its own span. No interpolation."""
    for i in range(len(y)):
        ax.plot([t0[i], t1[i]], [y[i], y[i]], **kw)
        kw.pop("label", None)


# --------------------------------------------------------------- R1
def fig_observed_surface(data_dir, path):
    """The observed incidence record as a (t, x) field, two honest renderings."""
    from scipy.interpolate import griddata
    panel = data_io.load_panel(data_dir, verbose=False)
    xs = np.asarray(panel.xg)[panel.gov]
    tm = 0.5 * (panel.t0 + panel.t1)
    y = panel.y

    fig = plt.figure(figsize=(13.4, 5.3))

    ax = fig.add_subplot(1, 2, 1, projection="3d")
    ti = np.linspace(tm.min(), tm.max(), 170)
    xi = np.linspace(xs.min(), xs.max(), 170)
    TI, XI = np.meshgrid(ti, xi)
    ZI = griddata((tm, xs), y, (TI, XI), method="linear")
    ax.plot_surface(TI, XI, ZI, cmap=FIELD_CMAP, linewidth=0, antialiased=True,
                    rcount=90, ccount=90)
    ax.set_xlabel("t  (days from 2017-05-22)"); ax.set_ylabel("x  (km from Sana'a)")
    ax.set_zlabel("incidence  (person$^{-1}$d$^{-1}$)")
    ax.set_title("(a)  observed $I(t,x)$, interpolated for display only", loc="left")
    ax.view_init(elev=26, azim=-124)

    ax = fig.add_subplot(1, 2, 2)
    sc = ax.scatter(tm, xs, c=y, s=13, cmap=FIELD_CMAP,
                    norm=LogNorm(vmin=max(y[y > 0].min(), 1e-8), vmax=y.max()),
                    edgecolor="none")
    ax.scatter(tm[y == 0], xs[y == 0], s=13, facecolor="none",
               edgecolor="#b0b0b0", linewidth=0.5, label="reported zero")
    for g, nm in enumerate(panel.names):
        ax.axhline(panel.xg[g], color="#000000", alpha=0.05, linewidth=0.5)
    ax.set_xlabel("t  (days)"); ax.set_ylabel("x  (km from Sana'a)")
    ax.set_title("(b)  every reported bulletin, nothing interpolated", loc="left")
    ax.legend(loc="upper right", fontsize=7.5, framealpha=0.9)
    fig.colorbar(sc, ax=ax, pad=0.02, label="incidence (person$^{-1}$d$^{-1}$)")
    fig.suptitle("Yemen cholera second wave, 21 reporting governorates, "
                 f"{len(y)} bulletin intervals", y=1.005, fontsize=10.5)
    fig.tight_layout(); fig.savefig(path); plt.close(fig)
    return panel


# --------------------------------------------------------------- R2 / R3
def fig_fit_and_error(R, path):
    """Observed against fitted, and the signed residual, for every city."""
    cities = list(R)
    n = len(cities)
    fig, axes = plt.subplots(2, n, figsize=(3.6 * n, 7.2), sharex=True)
    if n == 1:
        axes = axes.reshape(2, 1)
    for j, c in enumerate(cities):
        obs = R[c]["ff"]["observed"]
        t0, t1, y = obs["t0"], obs["t1"], obs["y"]
        ax = axes[0, j]
        _step(ax, t0, t1, y, color=C_INK, linewidth=1.6, label="observed",
              solid_capstyle="butt")
        for arm, nice, col, ls in ARMS:
            if arm not in R[c]:
                continue
            r = R[c][arm]["row"]
            _step(ax, t0, t1, R[c][arm]["yhat"], color=col, linewidth=1.15, ls=ls,
                  label=f"{nice}  MAE {r['MAE']:.2e}", solid_capstyle="butt")
        ax.set_title(c, loc="left")
        ax.set_yscale("symlog", linthresh=1e-6)
        if j == 0:
            ax.set_ylabel("incidence  (person$^{-1}$d$^{-1}$)")
        ax.legend(fontsize=6.6, loc="upper right", framealpha=0.9)

        ax = axes[1, j]
        ax.axhline(0, color=C_MUT, linewidth=0.7)
        for arm, nice, col, ls in ARMS:
            if arm not in R[c]:
                continue
            e = R[c][arm]["yhat"] - y
            r = R[c][arm]["row"]
            _step(ax, t0, t1, e, color=col, linewidth=1.1, ls=ls,
                  label=f"{nice}  RMSE {r['RMSE']:.2e}", solid_capstyle="butt")
        ax.set_xlabel("t  (days)")
        if j == 0:
            ax.set_ylabel("fitted $-$ observed")
        ax.legend(fontsize=6.6, loc="upper right", framealpha=0.9)
    fig.suptitle("Fitted against observed incidence, and the signed residual "
                 "(no reference solution is used)", y=1.003, fontsize=10.5)
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


# --------------------------------------------------------------- R4
def fig_beta(R, path):
    cities = list(R)
    n = len(cities)
    fig, axes = plt.subplots(2, n, figsize=(3.6 * n, 6.6), sharex=True)
    if n == 1:
        axes = axes.reshape(2, 1)
    for j, c in enumerate(cities):
        for row, key, lab in ((0, "beta10", r"$\beta_{1}(t)$  human route"),
                              (1, "beta20", r"$\beta_{2}(t)$  environmental route")):
            ax = axes[row, j]
            for arm, nice, col, ls in ARMS:
                if arm not in R[c]:
                    continue
                p = R[c][arm]
                ax.plot(p["t"], p[key], color=col, ls=ls, linewidth=1.2, label=nice)
            ax.set_yscale("log")
            if row == 0:
                ax.set_title(c, loc="left")
            if j == 0:
                ax.set_ylabel(lab)
            if row == 1:
                ax.set_xlabel("t  (days)")
            if row == 0 and j == n - 1:
                ax.legend(fontsize=7, loc="upper right", framealpha=0.9)
    fig.suptitle("Inferred time-dependent transmission rates. The two routes are "
                 "weakly identified: the data constrains their combination, not "
                 "the split.", y=1.004, fontsize=10.5)
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


# --------------------------------------------------------------- R5
def fig_R0(R, path):
    cities = list(R)
    n = len(cities)
    fig, axes = plt.subplots(1, n, figsize=(3.6 * n, 3.9), sharex=True)
    axes = np.atleast_1d(axes)
    for j, c in enumerate(cities):
        ax = axes[j]
        ax.axhline(1.0, color=C_INK, linewidth=0.9, ls=":", zorder=1)
        ax.axhspan(1.0, 5.0, color="#009E73", alpha=0.07, zorder=0)
        for arm, nice, col, ls in ARMS:
            if arm not in R[c]:
                continue
            p = R[c][arm]
            ax.plot(p["t"], p["R0"], color=col, ls=ls, linewidth=1.25,
                    label=f"{nice}  mean {p['row']['R0_mean']:.2f}")
        ax.set_yscale("log"); ax.set_title(c, loc="left")
        ax.set_xlabel("t  (days)")
        if j == 0:
            ax.set_ylabel(r"$\mathcal{R}_0(t)$")
        ax.legend(fontsize=6.8, loc="upper right", framealpha=0.9)
    fig.suptitle(r"Temporal reproduction number. Shaded band is the range "
                 r"reported for cholera ($1\leq\mathcal{R}_0\leq5$); the dotted "
                 r"line is the epidemic threshold.", y=1.01, fontsize=10.5)
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


# --------------------------------------------------------------- R6
def fig_error_bars(R, path):
    cities = list(R)
    arms = [a for a in ARMS if any(a[0] in R[c] for c in cities)]
    fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.1))
    w = 0.8 / len(arms)
    for ax, metric, lab in ((axes[0], "MAE", "mean absolute error"),
                            (axes[1], "RMSE", "root mean squared error")):
        for k, (arm, nice, col, _) in enumerate(arms):
            v = [R[c][arm]["row"][metric] if arm in R[c] else np.nan for c in cities]
            pos = np.arange(len(cities)) + (k - (len(arms) - 1) / 2) * w
            ax.bar(pos, v, width=w * 0.92, color=col, label=nice,
                   edgecolor="white", linewidth=0.6)
            for x, vv in zip(pos, v):
                if np.isfinite(vv):
                    ax.text(x, vv * 1.07, f"{vv:.1e}", ha="center", fontsize=6.2,
                            rotation=90, color=C_MUT)
        ax.set_yscale("log"); ax.set_xticks(np.arange(len(cities)))
        ax.set_xticklabels(cities, fontsize=8)
        ax.set_ylabel(f"{lab}  (person$^{{-1}}$d$^{{-1}}$)")
        ax.set_title(f"({'ab'[axes.tolist().index(ax)]})  {metric} against the "
                     f"observed record", loc="left")
        ax.grid(axis="x", visible=False)
        ax.legend(fontsize=7.4, framealpha=0.9)
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


# --------------------------------------------------------------- R7
def fig_age_surfaces(R, city, path):
    """The age-resolved infectious field each architecture reconstructs, and
    their difference. With no truth available the comparison is between the two
    architectures, and it is labelled as such."""
    have = [a for a in ("det", "dpinn", "ff") if a in R[city]]
    nice = {a: n for a, n, _, _ in ARMS}
    A = seip.A_MAX
    fig, axes = plt.subplots(1, len(have) + 1, figsize=(4.0 * (len(have) + 1), 3.9))
    surfs = {}
    for k, arm in enumerate(have):
        S = np.asarray(R[city][arm]["I_surface"])
        surfs[arm] = S
        t = R[city][arm]["t"]
        a = np.linspace(0, A, S.shape[1])
        m = axes[k].pcolormesh(t, a, S.T, cmap=FIELD_CMAP, shading="auto")
        axes[k].set_title(f"({'abcd'[k]})  {nice[arm]}   $I(t,a)$", loc="left")
        axes[k].set_xlabel("t  (days)")
        if k == 0:
            axes[k].set_ylabel("a  (days since infection)")
        fig.colorbar(m, ax=axes[k], pad=0.02)
    if "dpinn" in surfs and "ff" in surfs:
        d = surfs["ff"] - surfs["dpinn"]
        t = R[city]["ff"]["t"]; a = np.linspace(0, A, d.shape[1])
        v = np.abs(d).max()
        ax = axes[-1]
        m = ax.pcolormesh(t, a, d.T, cmap=DIVERGE, shading="auto",
                          norm=TwoSlopeNorm(vcenter=0.0, vmin=-v, vmax=v))
        ax.set_title(f"({'abcd'[len(have)]})  FF-PINN $-$ DPINN", loc="left")
        ax.set_xlabel("t  (days)")
        fig.colorbar(m, ax=ax, pad=0.02)
    fig.suptitle(f"{city}: age-resolved infectious field. The right panel is a "
                 "difference between architectures, not an error against a "
                 "known field.", y=1.01, fontsize=10.5)
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def summary_table(R):
    import pandas as pd
    rows = []
    for c in R:
        for arm, *_ in ARMS:
            if arm in R[c]:
                rows.append(R[c][arm]["row"])
    return pd.DataFrame(rows)


def main(data_dir="data_real", outdir="figures_real",
         cities=("Al Hudaydah", "Amanat Al Asimah", "Hajjah", "Amran")):
    os.makedirs(outdir, exist_ok=True)
    R = load(cities)
    fig_observed_surface(data_dir, f"{outdir}/R1_observed_surface.png")
    fig_fit_and_error(R, f"{outdir}/R2_fit_and_residual.png")
    fig_beta(R, f"{outdir}/R3_beta.png")
    fig_R0(R, f"{outdir}/R4_reproduction_number.png")
    fig_error_bars(R, f"{outdir}/R5_error_comparison.png")
    fig_age_surfaces(R, "Al Hudaydah", f"{outdir}/R6_age_surfaces.png")
    tab = summary_table(R)
    tab.to_csv(f"{outdir}/real_metrics.csv", index=False)
    cols = ["city", "method", "MAE", "RMSE", "nRMSE_rms", "R0_min", "R0_max",
            "R0_mean", "frac_t_R0_gt1", "env_share", "seconds"]
    import pandas as pd
    pd.set_option("display.width", 220)
    print(tab[cols].to_string(index=False, float_format=lambda v: f"{v:.4g}"))
    return R, tab


if __name__ == "__main__":
    main()

In [ ]:
%%writefile plots_domain.py
"""Full-domain (t,x) comparison on the real record, diffusion on."""
from __future__ import annotations
import pickle, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, TwoSlopeNorm, LinearSegmentedColormap
plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 170, "font.size": 9,
                     "axes.titlesize": 10, "axes.labelsize": 9, "axes.grid": True,
                     "grid.alpha": .25, "grid.linewidth": .5,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "figure.facecolor": "white", "savefig.bbox": "tight"})
DIV = LinearSegmentedColormap.from_list("d", ["#2166AC", "#7FA8CE", "#D9D9D9",
                                              "#E8977A", "#B2182B"])
ARMS = [("det", "Deterministic", "#666666", "-"),
        ("dpinn", "DPINN", "#0072B2", "--"),
        ("dpinn_fs", "DPINN-FS", "#56B4E9", "-."),
        ("fdpinn", "F-DPINN", "#009E73", ":"),
        ("ff", "FF-PINN", "#D55E00", "-")]
NOTE = {"det": " (stopped at max_nfev: NOT converged)"}


def load(root="domain_fits"):
    R = {}
    for a, *_ in ARMS:
        try:
            d = pickle.load(open(f"{root}/domain_{a}.pkl", "rb"))
        except FileNotFoundError:
            continue
        # the slimmed files are flat; the full ones nest the arm under "rec"
        if "rec" not in d:
            d = {**d, "rec": d}
        R[a] = d
    return R


def table(R):
    rows = []
    for a, nice, *_ in ARMS:
        if a not in R:
            continue
        d = R[a]; r = d["rec"]; y = d["panel_y"]; yh = np.asarray(r["yhat"]); e = yh - y
        rows.append(dict(arm=a, method=nice + NOTE.get(a, ""),
                         MAE=np.mean(np.abs(e)), MSE=np.mean(e ** 2),
                         RMSE=np.sqrt(np.mean(e ** 2)),
                         nRMSE=np.sqrt(np.mean(e ** 2)) / np.sqrt(np.mean(y ** 2)),
                         R0_min=r["R0"].min(), R0_max=r["R0"].max(),
                         R0_mean=r["R0"].mean(), seconds=r["seconds"]))
    return pd.DataFrame(rows)


def fig_domain(R, path):
    """Observed against fitted on the (t,x) plane, at the real bulletin points."""
    have = [a for a, *_ in ARMS if a in R]
    d0 = R[have[0]]
    tm = 0.5 * (d0["t0"] + d0["t1"]); xs = np.asarray(d0["xg"])[d0["gov"]]
    y = d0["panel_y"]
    pos = y[y > 0]
    nrm = LogNorm(vmin=max(pos.min(), 1e-9), vmax=y.max())
    n = len(have) + 1
    fig, axes = plt.subplots(2, n, figsize=(3.5 * n, 7.2), sharex=True, sharey=True)
    sc = axes[0, 0].scatter(tm, xs, c=np.maximum(y, nrm.vmin), s=9, cmap="viridis",
                            norm=nrm, edgecolor="none")
    axes[0, 0].set_title("(a)  observed", loc="left")
    axes[0, 0].set_ylabel("x  (km from Sana'a)")
    fig.colorbar(sc, ax=axes[0, 0], pad=.02)
    axes[1, 0].axis("off")
    for k, a in enumerate(have):
        yh = np.asarray(R[a]["rec"]["yhat"])
        nice = dict((x[0], x[1]) for x in ARMS)[a]
        ax = axes[0, k + 1]
        m = ax.scatter(tm, xs, c=np.maximum(yh, nrm.vmin), s=9, cmap="viridis",
                       norm=nrm, edgecolor="none")
        ax.set_title(f"({'abcdefg'[k+1]})  {nice} fitted", loc="left")
        fig.colorbar(m, ax=ax, pad=.02)
        e = yh - y; v = np.percentile(np.abs(e), 99)
        ax = axes[1, k + 1]
        m = ax.scatter(tm, xs, c=np.clip(e, -v, v), s=9, cmap=DIV,
                       norm=TwoSlopeNorm(0.0, -v, v), edgecolor="none")
        ax.set_title(f"      {nice} residual   MAE {np.mean(np.abs(e)):.2e}", loc="left")
        ax.set_xlabel("t  (days)")
        if k == 0:
            ax.set_ylabel("x  (km from Sana'a)")
        fig.colorbar(m, ax=ax, pad=.02)
    fig.suptitle(f"Full domain, 21 governorates, $D_1..D_4=50$ km$^2$/day, "
                 f"$x\\in[0,{d0['x_max']:.1f}]$ km — one $\\beta(t)$ shared across "
                 f"every location", y=1.003, fontsize=10.5)
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def fig_R0(R, path):
    fig, ax = plt.subplots(figsize=(7.2, 4.0))
    ax.axhline(1.0, color="#222", lw=.9, ls=":")
    ax.axhspan(1, 5, color="#009E73", alpha=.07, zorder=0)
    for a, nice, col, ls in ARMS:
        if a not in R:
            continue
        r = R[a]["rec"]
        ax.plot(R[a]["t"][:len(r["R0"])] if len(R[a]["t"]) >= len(r["R0"])
                else np.linspace(0, 272, len(r["R0"])), r["R0"],
                color=col, ls=ls, lw=1.3,
                label=f"{nice}  mean {r['R0'].mean():.2f}" + NOTE.get(a, ""))
    ax.set_yscale("log"); ax.set_xlabel("t  (days)")
    ax.set_ylabel(r"$\mathcal{R}_0(t)$")
    ax.set_title("Temporal reproduction number, full domain with diffusion on", loc="left")
    ax.legend(fontsize=7.2, framealpha=.9)
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


if __name__ == "__main__":
    import os
    os.makedirs("figures_real", exist_ok=True)
    R = load()
    t = table(R)
    pd.set_option("display.width", 200)
    print(t.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
    t.to_csv("figures_real/domain_metrics.csv", index=False)
    fig_domain(R, "figures_real/R7_domain_surface.png")
    fig_R0(R, "figures_real/R8_domain_R0.png")
    print("\nwrote figures_real/R7_domain_surface.png, R8_domain_R0.png")

In [ ]:
%%writefile plots_factorial.py
"""The 2x2 design: in-sample against held-out, and the attributed effects."""
from __future__ import annotations
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 170, "font.size": 9,
                     "axes.titlesize": 10, "axes.labelsize": 9, "axes.grid": True,
                     "grid.alpha": .25, "grid.linewidth": .5,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "figure.facecolor": "white", "savefig.bbox": "tight"})
ARMS = [("det", "Deterministic", "#666666"), ("dpinn", "DPINN", "#0072B2"),
        ("dpinn_fs", "DPINN-FS", "#56B4E9"), ("fdpinn", "F-DPINN", "#009E73"),
        ("ff", "FF-PINN", "#D55E00")]
CITIES = ["Al Hudaydah", "Amanat Al Asimah", "Hajjah", "Amran"]


def tables(csv="holdout_fits/factorial_metrics.csv"):
    d = pd.read_csv(csv)
    d = d[(d["mode"] == "interleave") & (d.tag.isna() | (d.tag == ""))]
    keys = [a for a, *_ in ARMS]
    ins = d.pivot_table(index="city", columns="arm", values="in_MAE").loc[CITIES, keys]
    out = d.pivot_table(index="city", columns="arm", values="out_MAE").loc[CITIES, keys]
    return d, ins, out


def effects(out):
    L = np.log(out[["dpinn", "dpinn_fs", "fdpinn", "ff"]])
    return pd.DataFrame({
        "Fourier states": 0.5 * ((L.fdpinn + L.ff) - (L.dpinn + L.dpinn_fs)),
        "Fourier beta":   0.5 * ((L.dpinn_fs + L.ff) - (L.dpinn + L.fdpinn)),
        "interaction":    0.5 * ((L.ff + L.dpinn) - (L.dpinn_fs + L.fdpinn))})


def fig(ins, out, path):
    E = effects(out)
    fig = plt.figure(figsize=(14.6, 4.4))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.15, 1.15, 1.0], wspace=.28)
    w = 0.16
    for col, (tab, lab) in enumerate(((ins, "in-sample MAE"), (out, "held-out MAE"))):
        ax = fig.add_subplot(gs[0, col])
        for k, (arm, nice, c) in enumerate(ARMS):
            ax.bar(np.arange(len(CITIES)) + (k - 2) * w, tab[arm].values,
                   width=w * .9, color=c, label=nice, edgecolor="white", lw=.5)
        ax.set_yscale("log"); ax.set_xticks(np.arange(len(CITIES)))
        ax.set_xticklabels(CITIES, fontsize=7.6, rotation=12)
        ax.set_ylabel("MAE  (person$^{-1}$d$^{-1}$)")
        ax.set_title(f"({'ab'[col]})  {lab}", loc="left")
        ax.grid(axis="x", visible=False)
        if col == 0:
            ax.legend(fontsize=6.9, ncol=2, framealpha=.9)
    lo = min(ins.values.min(), out.values.min()) * .6
    hi = max(ins.values.max(), out.values.max()) * 1.6
    for a in fig.axes:
        a.set_ylim(lo, hi)

    ax = fig.add_subplot(gs[0, 2])
    m = E.mean(); x = np.arange(3)
    cols = ["#009E73", "#D55E00", "#888888"]
    ax.bar(x, np.exp(m.values), color=cols, edgecolor="white", lw=.6)
    for xi, v in zip(x, np.exp(m.values)):
        ax.text(xi, v * 1.02, f"x{v:.2f}", ha="center", fontsize=8)
    for ci, city in enumerate(CITIES):
        ax.plot(x, np.exp(E.loc[city].values), "o", ms=3.4, color="#222",
                alpha=.65, label="per city" if ci == 0 else None)
    ax.axhline(1.0, color="#222", lw=.9, ls=":")
    ax.set_xticks(x); ax.set_xticklabels(E.columns, fontsize=8)
    ax.set_ylabel("factor on held-out MAE")
    ax.set_title("(c)  attributed effects", loc="left")
    ax.grid(axis="x", visible=False); ax.legend(fontsize=7)
    fig.suptitle("The Fourier-feature state embedding buys the in-sample fit and "
                 "causes the overfitting; the $\\beta$ basis does neither",
                 y=1.03, fontsize=10.5)
    fig.savefig(path, bbox_inches="tight"); plt.close(fig)


if __name__ == "__main__":
    import os
    os.makedirs("figures_real", exist_ok=True)
    d, ins, out = tables()
    pd.set_option("display.width", 220)
    print("IN-SAMPLE MAE\n", ins.to_string(float_format=lambda v: "%.3e" % v))
    print("\nHELD-OUT MAE\n", out.to_string(float_format=lambda v: "%.3e" % v))
    print("\nDEGRADATION out/in\n", (out / ins).to_string(float_format=lambda v: "%.1f" % v))
    E = effects(out)
    print("\nEFFECTS on log held-out MAE\n", E.to_string(float_format=lambda v: "%+.3f" % v))
    print("\nmean factor:", {k: round(float(v), 3) for k, v in np.exp(E.mean()).items()})
    r = d.pivot_table(index="city", columns="arm", values=["R0_min", "R0_max"])
    print("\nR0 range by arm")
    for a, nice, _ in ARMS:
        lo = r[("R0_min", a)]; hi = r[("R0_max", a)]
        print(f"  {nice:14s} " + "  ".join(f"{c[:11]}: {lo[c]:.2f}-{hi[c]:.2f}" for c in CITIES))
    fig(ins, out, "figures_real/R11_factorial.png")
    print("\nwrote figures_real/R11_factorial.png")

In [ ]:
import importlib, seip, data_io, fit, pinn, fit_one, plots_real
for m in (seip, data_io, fit, pinn, fit_one, plots_real):
    importlib.reload(m)
kappa, J1, J2 = seip.ngm_integrals(report=True)
print(f"\nnext-generation integrals: kappa_EI={kappa:.6f}  J1={J1:.6f}  J2={J2:.6f}")

## 3. The observed incidence field

Two renderings of the same rows, kept apart deliberately. Panel (a) interpolates the scattered
$(t, x, y)$ triples onto a mesh purely so a continuous surface can be drawn — nothing downstream
consumes it. Panel (b) plots only the $(t,x)$ pairs a governorate actually reported, with the
exact-zero reports drawn as open circles rather than dropped. Every number quoted anywhere else in
this notebook comes from the rows in (b).

In [ ]:
import plots_real
panel = plots_real.fig_observed_surface("data_real", "R1_observed_surface.png")
from IPython.display import Image, display
display(Image("R1_observed_surface.png"))
print("panel:", len(panel.y), "bulletin intervals over", len(panel.names), "governorates")

## 4. The four governorates fitted

Chosen by burden in the second wave. Al Hudaydah carries the largest absolute caseload and is the
coastal focus; Amanat Al Asimah is the capital; Hajjah is second by caseload; Amran carries the
highest attack rate of the four. Each is fitted as a single location, so the spatial operator is
switched off and $D_1=\dots=D_4=0$ — the reduction the per-city comparison is defined on.

In [ ]:
CITIES = ["Al Hudaydah", "Amanat Al Asimah", "Hajjah", "Amran"]
b = (cases.groupby("governorate")
          .agg(total_cases=("new_cases", "sum"), bulletins=("new_cases", "size"),
               population=("population", "first")))
b["attack_per_1000"] = 1000 * b.total_cases / b.population
display(b.loc[CITIES].sort_values("total_cases", ascending=False)
         .style.format({"total_cases": "{:,.0f}", "population": "{:,.0f}",
                        "attack_per_1000": "{:.1f}"}))

## 5. Fitting

Each (city, arm) runs as its own process writing its own result, so a long run cannot discard a
completed fit. On two cores a full pass is roughly 20 minutes per city; set `RUN_FITS = False` to
load results already on disk.

The three arms are:

- **Deterministic** — $\beta$ as an order-$M$ Fourier series, states obtained by a conventional
  forward solve of the PDE, objective the Poisson deviance. This is the conventional-solver
  cross-check: it satisfies the PDE exactly by construction.
- **DPINN** — dual network, soft physics.
- **FF-PINN** — Fourier-feature embedding, soft physics.

The objective is the Poisson deviance rather than weighted least squares because the record is
count data with a large zero fraction; under $1/\sqrt{\text{count}}$ weighting the exact-zero rows
carry most of the objective and drive the fit subcritical.

In [ ]:
FITS_B64 = """..."""

In [ ]:
import base64, io, tarfile, subprocess, sys, os, itertools

RUN_FITS = False      # True refits every arm from scratch (~20 min per city on 2 cores)

if RUN_FITS:
    for city, arm in itertools.product(CITIES, ["det", "dpinn", "ff"]):
        print(f"--- {city} / {arm} ---", flush=True)
        subprocess.run([sys.executable, "-u", "fit_one.py", "--city", city,
                        "--arm", arm, "--data-dir", "data_real",
                        "--outdir", "real_fits"], check=True)
    FIT_ROOT = "real_fits"
else:
    # the fits shipped with this notebook, reduced to the arrays the figures and
    # tables consume and cast to float32 (4.1 % of the full pickles)
    with tarfile.open(fileobj=io.BytesIO(base64.b64decode(FITS_B64)), mode="r:gz") as tf:
        tf.extractall(".")
    FIT_ROOT = "real_fits_slim"
    print("unpacked shipped fits into", FIT_ROOT)

R = plots_real.load(CITIES, root=FIT_ROOT)
print({c: sorted(R[c]) for c in R})

## 6. Fitted against observed, and the residual

The model is drawn on the same footing as the data: interval averages over each reported interval,
which is what the loss actually sees. Drawing an instantaneous rate against a 21-day interval
average would compare two different objects.

MAE and RMSE are both reported because they answer different questions on this record. The series
is spiky and zero-inflated, so RMSE is dominated by the batched-reporting spikes while MAE tracks
the typical interval; quoting one alone hides which it is.

In [ ]:
plots_real.fig_fit_and_error(R, "R2_fit_and_residual.png")
display(Image("R2_fit_and_residual.png"))

## 7. The inferred transmission rates

In [ ]:
plots_real.fig_beta(R, "R3_beta.png")
display(Image("R3_beta.png"))

## 8. Both temporal reproduction numbers

$$\mathcal{R}_0(t) \;=\; \bar S(t)\,\kappa_{EI}\,J_1\left[\beta_{1}(t) + \alpha_0\,\beta_{2}(t)\,J_2\right]$$

by the next-generation construction, with $\kappa_{EI}$, $J_1$ and $J_2$ the kernel integrals
printed in section 2.

In [ ]:
plots_real.fig_R0(R, "R4_reproduction_number.png")
display(Image("R4_reproduction_number.png"))

## 9. The comparison

In [ ]:
plots_real.fig_error_bars(R, "R5_error_comparison.png")
display(Image("R5_error_comparison.png"))
tab = plots_real.summary_table(R)
cols = ["city", "method", "MAE", "MSE", "RMSE", "nRMSE_rms",
        "R0_min", "R0_max", "R0_mean", "frac_t_R0_gt1", "env_share"]
display(tab[cols].style.format({c: "{:.4g}" for c in cols[2:]}))
tab.to_csv("real_metrics.csv", index=False)

In [ ]:
# the ratio that matters: how much the Fourier-feature embedding buys on the data term
piv = tab.pivot_table(index="city", columns="arm", values=["MAE", "RMSE", "nRMSE_rms"])
gain = pd.DataFrame({
    "MAE  DPINN / FF-PINN":  piv[("MAE", "dpinn")] / piv[("MAE", "ff")],
    "RMSE DPINN / FF-PINN":  piv[("RMSE", "dpinn")] / piv[("RMSE", "ff")],
    "MAE  determ. / FF-PINN": piv[("MAE", "det")] / piv[("MAE", "ff")],
    "nRMSE FF-PINN (frac of data rms)": piv[("nRMSE_rms", "ff")],
    "nRMSE DPINN  (frac of data rms)": piv[("nRMSE_rms", "dpinn")],
}).loc[CITIES]
display(gain.style.format("{:.2f}"))

## 10. The age-resolved field

With no true field available, the right-hand panel is a difference **between architectures**, not
an error against a known answer, and is labelled that way.

In [ ]:
plots_real.fig_age_surfaces(R, "Al Hudaydah", "R6_age_surfaces.png")
display(Image("R6_age_surfaces.png"))

## 11. Held out: is the in-sample advantage reconstruction or capacity?

Everything above is training error, and the FF-PINN carries capacity the other arms do not: its
states are free network outputs tied to the PDE only softly, so it can represent curves the PDE
itself cannot produce. That is the same property that leaves its $\mathcal{R}_0$ unconstrained. So
the advantage has to be measured out of sample.

Every fourth bulletin interval is held out, spread across the whole window, so $\beta$ interpolates
everywhere and the test isolates **generalisation**. 101 training intervals, 33 held out, identical
protocol for all three arms.

A forward split (train 0–200 d, predict 200–272 d) was tried first and discarded as the wrong
instrument: it degraded every arm by a factor above 200 and drove the deterministic
$\mathcal{R}_0$ to 45,185, because it extrapolates $\beta$ outside its data support where a
truncated Fourier series is unconstrained, and it leaves only 8 held-out intervals. It measures the
$\beta$ basis, not the architecture. It is kept below as a forecasting caveat.

In [ ]:
NEW_B64 = """..."""

In [ ]:
import base64, io, tarfile
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(NEW_B64)), mode="r:gz") as tf:
    tf.extractall(".")
H = pd.read_csv("new_slim/holdout_and_sweep.csv")
il = H[(H["mode"] == "interleave") & (H.tag.isna() | (H.tag == ""))]
piv = il.pivot_table(index="city", columns="arm", values=["in_MAE", "out_MAE"])
piv["degradation"] = piv[("out_MAE", "ff")] / piv[("in_MAE", "ff")]
order = CITIES
print("IN-SAMPLE MAE");  print(piv["in_MAE"].loc[order].to_string(float_format=lambda v: "%.3e" % v))
print("\nHELD-OUT MAE"); print(piv["out_MAE"].loc[order].to_string(float_format=lambda v: "%.3e" % v))
o = piv["out_MAE"].loc[order]
print("\nheld-out MAE relative to the deterministic arm  (>1 = worse than deterministic)")
print(o[["dpinn", "ff"]].div(o["det"], axis=0).to_string(float_format=lambda v: "%.2f" % v))
print("\nFF-PINN degradation out/in:",
      ", ".join(f"{c} x{piv['degradation'][c]:.1f}" for c in order))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.1)) if False else (None, None)
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.2))
w = 0.26
for ax, metric, lab in ((axes[0], "in_MAE", "in-sample MAE"),
                        (axes[1], "out_MAE", "held-out MAE")):
    for k, (arm, nice, col) in enumerate([("det", "Deterministic", "#666666"),
                                          ("dpinn", "DPINN", "#0072B2"),
                                          ("ff", "FF-PINN", "#D55E00")]):
        v = [float(il[(il.city == c) & (il.arm == arm)][metric].iloc[0]) for c in CITIES]
        ax.bar(np.arange(len(CITIES)) + (k - 1) * w, v, width=w * .92, color=col,
               label=nice, edgecolor="white", linewidth=.6)
    ax.set_yscale("log"); ax.set_xticks(np.arange(len(CITIES)))
    ax.set_xticklabels(CITIES, fontsize=8); ax.set_ylabel(lab)
    ax.set_title(lab, loc="left"); ax.grid(axis="x", visible=False)
    ax.legend(fontsize=7.4)
axes[0].set_ylim(axes[1].get_ylim())
fig.suptitle("The FF-PINN's in-sample margin does not survive a holdout", y=1.02, fontsize=10.5)
fig.tight_layout(); fig.savefig("R9_holdout.png", dpi=170, bbox_inches="tight")
display(Image("R9_holdout.png"))

## 12. Does stiffer physics enforcement restore $\mathcal{R}_0$?

Both the deterministic arm and the FF-PINN use $\beta$ as an order-4 Fourier series — the
parameterisation is identical. So the $\mathcal{R}_0$ divergence between them must come from the
remaining difference: the deterministic arm's states come from a forward solve under a hard
constraint, the network's are free functions under a soft penalty. The natural prediction is that
raising $w_{\text{pde}}$ moves the FF-PINN toward the deterministic answer.

It does the opposite, and the negative result is the more informative one.

In [ ]:
sw = H[H.tag.astype(str).str.startswith("wpde")].copy()
sw["w_pde"] = sw.w_pde.astype(float)
sw = sw.sort_values("w_pde")
print(sw[["w_pde", "in_MAE", "out_MAE", "R0_min", "R0_max", "R0_mean"]]
      .to_string(index=False, float_format=lambda v: "%.4g" % v))

fig, ax1 = plt.subplots(figsize=(7.4, 4.2))
ax1.plot(sw.w_pde, sw.in_MAE, "o-", color="#D55E00", label="in-sample MAE")
ax1.plot(sw.w_pde, sw.out_MAE, "s--", color="#0072B2", label="held-out MAE")
ax1.set_xscale("log"); ax1.set_yscale("log")
ax1.set_xlabel(r"$w_{\mathrm{pde}}$"); ax1.set_ylabel("MAE")
ax2 = ax1.twinx()
ax2.plot(sw.w_pde, sw.R0_max, "^-", color="#009E73", label=r"max $\mathcal{R}_0$")
ax2.set_yscale("log"); ax2.set_ylabel(r"max $\mathcal{R}_0(t)$"); ax2.grid(False)
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, fontsize=7.6, loc="upper left")
ax1.set_title("Al Hudaydah: stiffening the physics does not recover the reproduction number",
              loc="left", fontsize=9.5)
fig.tight_layout(); fig.savefig("R10_wpde_sweep.png", dpi=170, bbox_inches="tight")
display(Image("R10_wpde_sweep.png"))

Across five orders of magnitude the data error moves by a factor of 2.2 and the held-out error by
1.5 with no monotone trend, while the upper end of $\mathcal{R}_0$ rises monotonically from 6.8 to
74.1. Enforcing the PDE harder makes the reproduction number *worse*.

The reason is that the amplitude degeneracy
$(E,I,P,\beta_{10},\beta_{20}) \to (cE,cI,cP,\beta_{10}/c,\beta_{20}/c)$ is a symmetry of the PDE
residual itself: the equations are linear in the states given $\beta$ and the renewal condition is
bilinear, so rescaling leaves the residual invariant. Weighting that residual more heavily presses
the solution onto the degenerate manifold without selecting a point on it. Only the initial
condition and the data break the symmetry. **$\mathcal{R}_0$ non-identifiability is a property of
the observation operator, not of how the solver is enforced.**

## 13. The spatial model, with diffusion on

Sections 5–12 fit isolated locations with $D_1=\dots=D_4=0$, while the model is written as a
transport–diffusion system on $(t,x)$. The full-domain fit is run here: $x\in[0,799.385]$ km, all
21 reporting governorates, 2,785 intervals, $D_1..D_4 = 50$ km²/day, and **one $\beta(t)$ shared
across every location**.

In [ ]:
import plots_domain, importlib
importlib.reload(plots_domain)
RD = plots_domain.load("new_slim")
td = plots_domain.table(RD)
display(td.style.format({c: "{:.4g}" for c in
                         ["MAE", "MSE", "RMSE", "nRMSE", "R0_min", "R0_max", "R0_mean", "seconds"]}))
plots_domain.fig_domain(RD, "R7_domain_surface.png")
plots_domain.fig_R0(RD, "R8_domain_R0.png")
display(Image("R7_domain_surface.png")); display(Image("R8_domain_R0.png"))

In [ ]:
# where the domain error actually sits, by governorate
d = RD["ff"]; y = d["panel_y"]; gov = d["gov"]; xg = np.asarray(d["xg"])
rows = []
for g, nm in enumerate(d["names"]):
    m = gov == g
    r = dict(governorate=nm, x_km=xg[g], n=int(m.sum()), obs_mean=y[m].mean())
    for a in ("det", "dpinn", "ff"):
        yh = np.asarray(RD[a]["yhat"])[m]
        r[f"{a}_MAE"] = np.mean(np.abs(yh - y[m])); r[f"{a}_bias"] = np.mean(yh - y[m])
    rows.append(r)
gt = pd.DataFrame(rows).sort_values("x_km")
display(gt.style.format({c: "{:.3e}" for c in gt.columns if c not in ("governorate", "n", "x_km")}))
far = gt[gt.x_km > 400]
for a in ("det", "dpinn", "ff"):
    num = (far[f"{a}_MAE"] * far.n).sum(); den = (gt[f"{a}_MAE"] * gt.n).sum()
    print(f"{a:6s}: Hadramaut + Al Maharah are {100*far.n.sum()/gt.n.sum():.1f} % of rows "
          f"and carry {100*num/den:.1f} % of total |error|")

Three things change once the model is spatial.

**The FF-PINN's margin collapses** to 1.6× on MAE against the DPINN, from 5.3–8.5× in the isolated
fits. With one $\beta(t)$ explaining 21 governorates at once, the extra flexibility has far less
room to absorb noise.

**Everything fits worse** — nRMSE 0.45–0.67 against 0.03–0.50 per city. Part of the per-city
performance was each governorate receiving a transmission history of its own.

**Identifiability improves.** The DPINN's $\mathcal{R}_0$ falls from 3.24–16.31 at Al Hudaydah in
isolation to 0.87–5.37 over the whole domain; the FF-PINN's upper end falls from 49.74 at Hajjah to
11.18. Sharing $\beta$ across 21 locations partially breaks the amplitude degeneracy that no amount
of physics weighting could touch — which points at the remedy section 12 rules out.

On the residual panel the two far governorates look grossly over-predicted. The cell above shows
they are not: Hadramaut and Al Maharah carry 1.5 % (DPINN), 1.7 % (FF-PINN) and 6.4 %
(deterministic) of total absolute error against an 8.7 % share of rows. The logarithmic colour
floor exaggerates them. The error concentrates in the mid-range high-incidence governorates, and
the informative pattern there is systematic per-governorate **bias** in the deterministic arm —
Amran −2.0e-04, Abyan −2.0e-04, Al Mahwit −1.7e-04 — against FF-PINN biases roughly an order of
magnitude smaller. A single shared $\beta(t)$ cannot express the observed spatial heterogeneity;
the network absorbs into its free states what the shared parameter cannot. That is an argument for
a spatially varying $\beta(t,x)$, which this methodology does not currently have.

## 14. The 2x2 factorial: separating the two design factors

Sections 5-13 compare two arms that differ in **two** ways at once. `kind="dpinn"` set the state
networks to plain MLPs *and* $\beta$ to a free MLP; `kind="ff"` set Fourier-feature embeddings *and*
$\beta$ to an order-4 truncated series. Neither effect is attributable from that pair. The flag is
now split into independent `state_ff` and `beta_kind` factors (see `ARCH` in `pinn.py`), completing
the design:

|                          | $\beta$ = free MLP | $\beta$ = order-4 Fourier series |
|--------------------------|--------------------|----------------------------------|
| states: plain MLP        | DPINN              | **DPINN-FS** |
| states: Fourier features | **F-DPINN**        | FF-PINN |

Same four cities, same interleaved holdout, same budget and seed in every cell. Predictions were
recorded before the two new cells were fitted; the scorecard closes this section.

In [ ]:
import plots_factorial, importlib
importlib.reload(plots_factorial)
d, ins, out = plots_factorial.tables("new_slim/holdout_and_sweep.csv")
print("IN-SAMPLE MAE\n", ins.to_string(float_format=lambda v: "%.3e" % v))
print("\nHELD-OUT MAE\n", out.to_string(float_format=lambda v: "%.3e" % v))
print("\nDEGRADATION out/in\n", (out / ins).to_string(float_format=lambda v: "%.1f" % v))
E = plots_factorial.effects(out)
print("\nEFFECTS on log held-out MAE (negative lowers error)\n",
      E.to_string(float_format=lambda v: "%+.3f" % v))
print("\nmean multiplicative factor:",
      {k: round(float(v), 3) for k, v in np.exp(E.mean()).items()})
plots_factorial.fig(ins, out, "R11_factorial.png")
display(Image("R11_factorial.png"))

**The state embedding buys the entire in-sample gain and causes the entire overfitting.** Moving
from plain to Fourier-feature states cuts in-sample MAE by 10-85x at every city and raises
degradation from 0.9-1.5 to 8.7-53.1. Out of sample it is neutral at best: x1.12 on average, and
x1.30-1.72 at three of four cities.

**The $\beta$ basis is close to inert**, x0.93 on held-out error. The full-domain fit settles it,
because there all five arms fit the same 2,785 intervals: the two plain-state arms land at nRMSE
0.6261 and 0.6259, the two Fourier-state arms at 0.4453 and 0.4463. Changing $\beta$ moves the
fourth significant figure; changing the state embedding moves it by 1.4x.

**DPINN-FS, absent from the original study, is the most robust cell** - first or second on held-out
error at all four cities, degradation 0.9-1.4x, never worse than 1.01x the deterministic baseline,
and the cheapest of the four to train.

## 15. All five arms on the $(t,x)$ surface

In [ ]:
RD = plots_domain.load("new_slim")
display(plots_domain.table(RD).style.format(
    {c: "{:.4g}" for c in ["MAE", "MSE", "RMSE", "nRMSE", "R0_min", "R0_max", "R0_mean", "seconds"]}))
plots_domain.fig_domain(RD, "R7_domain_surface.png")
plots_domain.fig_R0(RD, "R8_domain_R0.png")
display(Image("R7_domain_surface.png")); display(Image("R8_domain_R0.png"))

Panels (c) DPINN and (d) DPINN-FS are visually indistinguishable, as are (e) F-DPINN and (f)
FF-PINN. The $\beta$ basis leaves no visible trace on the fitted surface; the state embedding
changes it.

### Scorecard against the recorded predictions

| # | prediction | outcome |
|---|---|---|
| 1 | F-DPINN lowest in-sample of the four | **mostly held** - lowest at 3 of 4; FF-PINN edges it at Al Hudaydah |
| 2 | F-DPINN never beats the deterministic arm out of sample | **falsified** at Al Hudaydah (0.63x); held at the other three |
| 3 | the $\beta$ parameterisation is the dominant factor | **falsified, and backwards** - the state embedding dominates both effects |
| 4 | held-out error flat across cells within ~2x | held, except DPINN at Al Hudaydah (2.66x) |
| 5 | Fourier-$\beta$ cells give narrower $\mathcal{R}_0$ | **not supported** - F-DPINN (MLP $\beta$) is narrower than FF-PINN at 3 of 4 |

Three of five were wrong, and prediction 3 was wrong in the informative direction: the earlier
three-arm result suggested the low-order $\beta$ series was doing the regularising. It does almost
nothing. The state embedding is simultaneously the source of the apparent gain and of the
overfitting - an attribution that was not recoverable from the confounded two-arm comparison.

## 16. What the comparison shows

**The Fourier-feature embedding wins the data term in sample, and that is not the result.** FF-PINN
cuts in-sample MAE by 5.3–8.5× over DPINN at every governorate. Held out, it loses to the
deterministic four-term Fourier series at three of four governorates (1.27×, 1.38×, 1.77× worse)
and ties at the fourth (0.96×). Its degradation is 16.6–22.6× and uniform across cities: it is
systematically about twentyfold optimistic about its own error. **Neither network generalises
better than a low-order deterministic fit under a hard PDE constraint.**

**The dual network barely improves on that Fourier series even in sample.** At Hajjah and Amran its
normalised RMSE (0.486, 0.316) is indistinguishable from the deterministic arm's (0.495, 0.315) —
the spectral-bias failure the Fourier-feature construction exists to repair, observed on real data.

**That failure is architectural, not a tuning artifact.** Doubling the DPINN's Adam budget and
raising the renewal coupling fivefold at Al Hudaydah left the data loss unchanged (65.1 against
65.6) and widened $\mathcal{R}_0$ from 3.24–16.31 to 2.52–60.08.

**$\mathcal{R}_0$ and the route split are not identified by this record, and physics weighting
cannot fix it.** Environmental share spans 0.014–0.33 deterministic against 0.70–0.99 for both
PINNs on identical data, and the $w_{\text{pde}}$ sweep shows the non-identifiability is a symmetry
of the observation operator that stiffer enforcement presses into rather than removes. Spatial
coupling is the thing that helps.

**What follows for the paper.** The defensible claims are: that on aggregated bulletin counts at a
single location neither PINN architecture beats a low-order deterministic fit out of sample, and
in-sample residuals would have said otherwise by a factor of twenty; that $\mathcal{R}_0$ and the
route split are unidentified by the observation operator regardless of how the physics is enforced;
and that sharing $\beta$ across locations is what partially restores identifiability. Reporting the
FF-PINN's $\mathcal{R}_0$ as an estimate is not defensible.

**Limits, stated rather than implied.** One seed per architecture, so between-arm differences are
not yet separated from initialisation variance — the first thing a referee will ask for. The
holdout is a single deterministic every-fourth split, not repeated random splits. The deterministic
full-domain arm stopped at `max_nfev` and is reported as not converged; its domain row is a lower
bound on that arm's attainable fit. The physics sweep is one city. Diffusion is fixed at
50 km²/day and never estimated. Second wave only, and no comparison against an independently
published $R_t$ reconstruction for Yemen.